## PART 3 of 3 — Full Evaluation, XAI, Ablation & All Figures

> **Requires:** Add Part 1 output as `alzheimer-part1-output` AND Part 2 output as `alzheimer-part2-output`

# Alzheimer's Disease Detection — Full Research Pipeline
## Dual-Channel ConvNeXt-Tiny + Swin-V2-T + CBAM | FedBN Federated Learning

Architecture : ConvNeXt-Tiny (CNN) + Swin-V2-T (Transformer) + CBAM
Loss         : WCE(label_smoothing=0.1) + Focal + Supervised Contrastive Loss
Tuning       : Optuna TPE sampler + MedianPruner (20 trials)
Validation   : 5-Fold Patient-Level Stratified Cross-Validation
Federated    : FedAvg | FedProx | FedBN
XAI          : Grad-CAM++ | Score-CAM | SHAP | LIME | Attention Rollout
Analysis     : Ablation (13 variants) | t-SNE/UMAP | Calibration |
               Uncertainty (MC-Dropout) | Statistical Tests | Robustness |
               Fairness | Error Analysis | Model Saving

Datasets:
  Client 1 (Primary)  : ninadaithal/imagesoasis  (OASIS-1, ~80 K slices, patient IDs)
  Client 2 (External) : uraninjo /original/       (6 400 images, external test)



## Step 1 -- Clinical Task Definition & Label Mapping

Fulfils **Proposal Step 1**: define the classification task and document
how every dataset label maps to the unified label set.
All downstream code uses `CLASS2IDX` -- never raw string labels.


In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import os, time
KAGGLE_START_TIME = time.time()
MAX_KAGGLE_SECONDS = 11.3 * 3600  # 11 hours 18 mins safety cut-off

def check_timeout(label=""):
    elapsed = time.time() - KAGGLE_START_TIME
    remaining = MAX_KAGGLE_SECONDS - elapsed
    if remaining < 600:  # less than 10 min left
        print(f"[Timeout Guard] WARNING: Only {remaining/60:.1f} min remaining! Saving and stopping at: {label}")
        return True
    return False

print(f"[Timeout Guard] Active. Will stop after 11.3 hours.")


[Timeout Guard] Active. Will stop after 11.3 hours.


In [2]:
# ====================================================================
# STEP 1 -- CLINICAL TASK DEFINITION & LABEL MAPPING
# Proposal requirement: define ONE label dictionary; keep it identical
# across all splits, datasets, and evaluation stages.
# ====================================================================

import textwrap

# 1a. Unified class set (4-class multi-stage task)
CLASSES     = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(CLASSES)

# 1b. Clinical stage descriptions
CLINICAL_DESC = {
    'NonDemented':       'Cognitively normal; CDR = 0',
    'VeryMildDemented':  'Very mild cognitive impairment; CDR = 0.5',
    'MildDemented':      'Mild dementia; CDR = 1',
    'ModerateDemented':  'Moderate dementia; CDR = 2',
}

# 1c. Dataset-specific label harmonisation maps
LABEL_MAP_OASIS = {
    'Non Demented':       'NonDemented',
    'NonDemented':        'NonDemented',
    'Very Mild Demented': 'VeryMildDemented',
    'VeryMildDemented':   'VeryMildDemented',
    'Mild Demented':      'MildDemented',
    'MildDemented':       'MildDemented',
    'Moderate Demented':  'ModerateDemented',
    'ModerateDemented':   'ModerateDemented',
}

LABEL_MAP_EXTERNAL = {
    'NonDemented':       'NonDemented',
    'VeryMildDemented':  'VeryMildDemented',
    'MildDemented':      'MildDemented',
    'ModerateDemented':  'ModerateDemented',
}

# 1d. Task description
TASK_NAME        = "4-class Alzheimer's disease multi-stage classification"
TASK_DESCRIPTION = textwrap.dedent("""
    Task  : Multi-stage AD detection (4 classes, Proposal Step 1).
    Labels: NonDemented | VeryMildDemented | MildDemented | ModerateDemented
    Metric: AUC-macro (primary), F1-macro, Balanced Accuracy (secondary).
    Split : Patient-level stratified split -- zero data leakage.
    Data  : OASIS (internal train/val/test) + External (external test).
""").strip()

# 1e. Print label registry
print('=' * 65)
print(f'  TASK: {TASK_NAME}')
print('=' * 65)
print(f'{"Index":<8} {"Class Name":<22} {"Clinical Description"}')
print('-' * 65)
for cls, idx in CLASS2IDX.items():
    print(f'  {idx:<6} {cls:<22} {CLINICAL_DESC[cls]}')
print()
print('OASIS label mapping:')
for src_lbl, tgt_lbl in LABEL_MAP_OASIS.items():
    if src_lbl != tgt_lbl:
        print(f"  '{src_lbl}'  ->  '{tgt_lbl}'  (idx={CLASS2IDX[tgt_lbl]})")
print()
print('External dataset label mapping:')
for src_lbl, tgt_lbl in LABEL_MAP_EXTERNAL.items():
    print(f"  '{src_lbl}'  ->  idx={CLASS2IDX[tgt_lbl]}")
print()
print(TASK_DESCRIPTION)


  TASK: 4-class Alzheimer's disease multi-stage classification
Index    Class Name             Clinical Description
-----------------------------------------------------------------
  0      NonDemented            Cognitively normal; CDR = 0
  1      VeryMildDemented       Very mild cognitive impairment; CDR = 0.5
  2      MildDemented           Mild dementia; CDR = 1
  3      ModerateDemented       Moderate dementia; CDR = 2

OASIS label mapping:
  'Non Demented'  ->  'NonDemented'  (idx=0)
  'Very Mild Demented'  ->  'VeryMildDemented'  (idx=1)
  'Mild Demented'  ->  'MildDemented'  (idx=2)
  'Moderate Demented'  ->  'ModerateDemented'  (idx=3)

External dataset label mapping:
  'NonDemented'  ->  idx=0
  'VeryMildDemented'  ->  idx=1
  'MildDemented'  ->  idx=2
  'ModerateDemented'  ->  idx=3

Task  : Multi-stage AD detection (4 classes, Proposal Step 1).
Labels: NonDemented | VeryMildDemented | MildDemented | ModerateDemented
Metric: AUC-macro (primary), F1-macro, Balanced Accuracy

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
## Section 1 — Imports & Global Configuration

In [3]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import os, re, random, copy, json, time, warnings
import copy   
import json   
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, fbeta_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, brier_score_loss,
    precision_score, recall_score, matthews_corrcoef,
)
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon

import timm
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")


SEED         = 42
NUM_CLASSES  = 4
CLASSES      = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
CLASS2IDX    = {c: i for i, c in enumerate(CLASSES)}


DATA_ROOT     = "/kaggle/input/datasets"                                              
OASIS_ROOT    = f"{DATA_ROOT}/ninadaithal/imagesoasis/Data"
EXTERNAL_ROOT = f"{DATA_ROOT}/uraninjo/augmented-alzheimer-mri-dataset/OriginalDataset"


OASIS_FOLDER_MAP = {
    "Non Demented"       : "NonDemented",
    "Mild Dementia"      : "MildDemented",
    "Moderate Dementia"  : "ModerateDemented",
    "Very mild Dementia" : "VeryMildDemented",
}

IMG_SIZE       = 256
BATCH_SIZE      = 32         # safe for kaggle P100/T4 OOM         # larger batch -> smoother gradients
NUM_WORKERS     = 2
OUTER_FOLDS     = 3
OPTUNA_TRIALS   = 10   # reduced: 15→5 (TPE finds good params fast)         # more trials -> better hyperparams
FL_ROUNDS       = 20  # reduced: 20->10 (convergence happens in first 10 rounds)         # more FL rounds -> better convergence
FL_LOCAL_EPOCHS = 5  # increased from 3 for better convergence
FEDPROX_MU      = 0.01

# Training recipe for ~96% accuracy
MAX_EPOCHS      = 25         # more epochs (early stopping prevents overfit)
WARMUP_EPOCHS   = 5          # LR warmup to stabilise early training
BASE_LR         = 2e-4       # higher initial LR with warmup
MIN_LR          = 1e-6       # cosine annealing floor
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1        # prevents overconfidence
EMA_DECAY       = 0.9998     # Exponential Moving Average for smooth curves
GRAD_CLIP       = 1.0        # gradient clipping
MIXUP_ALPHA     = 0.2        # Mixup augmentation
PATIENCE        = 8         # more patience with cosine schedule

SAVE_DIR = Path("./outputs")
for sub in ["models", "figures", "results", "xai"]:
    (SAVE_DIR / sub).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)
print(f"Device : {device}")
print(f"Classes: {CLASSES}")

Device : cuda
Classes: ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']


## Section 2 — Data Loading & Patient-Level Split

In [4]:
def extract_patient_id(filename: str) -> str:
    
    m = re.match(r"(OAS\d+_\d+_MR\d+)", filename)
    return m.group(1) if m else os.path.splitext(filename)[0]

def build_df_oasis(root: str) -> pd.DataFrame:
    """
    Load OASIS dataset.
    OASIS_FOLDER_MAP keys = actual disk folder names on Kaggle.
    Falls back to class name if disk name not found.
    """
    rows = []
    if not os.path.isdir(root):
        print(f'[WARN] OASIS root not found: {root}')
        return pd.DataFrame(columns=['path','label','class_name','patient_id','source'])
    for disk_name, class_name in OASIS_FOLDER_MAP.items():
        folder = os.path.join(root, disk_name)
        if not os.path.isdir(folder):
            folder = os.path.join(root, class_name)   # fallback: try class name
        if not os.path.isdir(folder):
            print(f'[WARN] OASIS: folder not found: {disk_name} or {class_name}')
            continue
        lbl = CLASS2IDX[class_name]
        for fn in os.listdir(folder):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append({'path': os.path.join(folder, fn),
                             'label': lbl, 'class_name': class_name,
                             'patient_id': extract_patient_id(fn),
                             'source': 'oasis'})
    df = pd.DataFrame(rows)
    print(f'[OASIS] {len(df)} images | {df["patient_id"].nunique()} patients')
    print(df['class_name'].value_counts())
    return df

def build_df_external(root: str) -> pd.DataFrame:
    """
    Load external dataset.
    Uses case-insensitive folder matching to handle 'Nondemented' vs 'NonDemented'.
    """
    rows = []
    # Build case-insensitive lookup: lowercase_folder_name -> actual_path
    if not os.path.isdir(root):
        print(f'[WARN] External root not found: {root}')
        return pd.DataFrame(columns=['path','label','class_name','patient_id','source'])
    folder_map = {f.lower(): f for f in os.listdir(root)
                  if os.path.isdir(os.path.join(root, f))}
    for class_name in CLASSES:
        # Try exact match first, then case-insensitive
        actual_folder = folder_map.get(class_name.lower())
        if actual_folder is None:
            print(f'[WARN] External: no folder for {class_name}'); continue
        folder = os.path.join(root, actual_folder)
        lbl = CLASS2IDX[class_name]
        for fn in os.listdir(folder):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append({'path': os.path.join(folder, fn),
                             'label': lbl, 'class_name': class_name,
                             'patient_id': f'ext_{fn}',
                             'source': 'external'})
    df = pd.DataFrame(rows)
    print(f'[External] {len(df)} images')
    if len(df) > 0:
        print(df['class_name'].value_counts())
    return df

def patient_level_split(df, test_size=0.15, val_size=0.15, seed=SEED):
    """Split by patient_id — zero leakage guaranteed."""
    pid_label = (df.groupby("patient_id")["label"]
                   .agg(lambda x: x.mode()[0]).reset_index()
                   .rename(columns={"label": "plabel"}))
    pids, plabels = pid_label["patient_id"].values, pid_label["plabel"].values

    pids_tr, pids_te = train_test_split(
        pids, test_size=test_size, stratify=plabels, random_state=seed)
    plabels_tr = pid_label.set_index("patient_id").loc[pids_tr, "plabel"].values
    pids_tr, pids_va = train_test_split(
        pids_tr, test_size=val_size/(1-test_size),
        stratify=plabels_tr, random_state=seed)

    df_tr = df[df["patient_id"].isin(pids_tr)].reset_index(drop=True)
    df_va = df[df["patient_id"].isin(pids_va)].reset_index(drop=True)
    df_te = df[df["patient_id"].isin(pids_te)].reset_index(drop=True)
    print(f"Train {len(df_tr)} | Val {len(df_va)} | Test {len(df_te)}")
    return df_tr, df_va, df_te

df_oasis    = build_df_oasis(OASIS_ROOT)
df_external = build_df_external(EXTERNAL_ROOT)
df_train, df_val, df_test_internal = patient_level_split(df_external)
df_test_external = df_oasis   


print("\n── OASIS dataset ──")
print(df_oasis["label"].value_counts().rename(index=dict(enumerate(CLASSES))))
print(f"Total OASIS images : {len(df_oasis)}")
display(df_oasis.head(3))

print("\n── External dataset ──")
print(df_external["label"].value_counts().rename(index=dict(enumerate(CLASSES))))
print(f"Total External images : {len(df_external)}")
display(df_external.head(3))

print(f"\nSplit → Train:{len(df_train)} | Val:{len(df_val)} "
      f"| Test-Internal:{len(df_test_internal)} | Test-External:{len(df_test_external)}")

[OASIS] 86437 images | 366 patients
class_name
NonDemented         67222
VeryMildDemented    13725
MildDemented         5002
ModerateDemented      488
Name: count, dtype: int64
[External] 6400 images
class_name
NonDemented         3200
VeryMildDemented    2240
MildDemented         896
ModerateDemented      64
Name: count, dtype: int64
Train 4489 | Val 960 | Test 951

── OASIS dataset ──
label
NonDemented         67222
VeryMildDemented    13725
MildDemented         5002
ModerateDemented      488
Name: count, dtype: int64
Total OASIS images : 86437


,path,label,class_name,patient_id,source
0,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0302_MR1,oasis
1,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0114_MR1,oasis
2,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0150_MR1,oasis



── External dataset ──
label
NonDemented         3200
VeryMildDemented    2240
MildDemented         896
ModerateDemented      64
Name: count, dtype: int64
Total External images : 6400


,path,label,class_name,patient_id,source
0,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem706.jpg,external
1,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem366.jpg,external
2,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem222.jpg,external



Split → Train:4489 | Val:960 | Test-Internal:951 | Test-External:86437


### Checkpoint Save -- After Section 2 (Data)
Saves all DataFrames. **Run this once after Section 2 completes.**
After saving, you can reload them via the Resume Loader to skip Section 2 next time.


In [5]:
# CHECKPOINT: Save DataFrames after Section 2
import pickle
from pathlib import Path
CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

_data_ckpt = {
    'df_oasis':         df_oasis,
    'df_external':      df_external,
    'df_train':         df_train,
    'df_val':           df_val,
    'df_test_internal': df_test_internal,
    'df_test_external': df_test_external,
}
with open(CKPT_DIR / 'ckpt_data.pkl', 'wb') as _f:
    pickle.dump(_data_ckpt, _f, protocol=pickle.HIGHEST_PROTOCOL)
print('[CHECKPOINT SAVED] ckpt_data.pkl')
print(f'  train={len(df_train)} | val={len(df_val)} | '
      f'test_internal={len(df_test_internal)} | test_external={len(df_test_external)}')


[CHECKPOINT SAVED] ckpt_data.pkl
  train=4489 | val=960 | test_internal=951 | test_external=86437


## Section 3 — Preprocessing & Augmentation

## Section 3a -- Deep MRI Preprocessing (Proposal Step 5)

Standard resize+normalise is **insufficient** for a CVPR-level pipeline.  
This section adds:
- **Skull-strip simulation** (elliptical mask removes non-brain pixels)
- **Bias-field correction** (N4-style Gaussian low-freq field removal)
- **CLAHE** adaptive contrast enhancement
- **Gaussian denoising** (acquisition noise reduction)
- **ROI/Hippocampus-guided cropping** (medial temporal lobe focus)

All steps are applied to **both train and eval** transforms;
augmentation (flips, jitter, erasing) is **training-only**.


In [6]:
# ====================================================================
# STEP 5 -- NOVEL AND ROBUST MRI PREPROCESSING (SAFE VERSION)
# Fixed: sigma, skull-strip safety, correct order, no image destruction
# ====================================================================

import cv2
import numpy as np
from PIL import Image


class SkullStripSimulation:
    """
    SAFE skull-strip simulation for 2D MRI slices.
    Uses a SOFT elliptical weight (not hard zero) so brain tissue at
    the edges is attenuated rather than destroyed.
    Kaggle/OASIS 2D slices are already preprocessed, so this is
    applied with blend_strength=0.1 to avoid damaging existing features.
    Set enabled=False to disable entirely if images are pre-stripped.
    """
    def __init__(self, blend_strength=0.1, enabled=True):  # ↓ was 0.3 — lighter strip preserves more features
        self.blend   = blend_strength
        self.enabled = enabled

    def __call__(self, img):
        if not self.enabled:
            return img
        try:
            arr = np.array(img.convert('RGB')).astype(np.float32)
            h, w = arr.shape[:2]
            # Soft Gaussian weight mask centred on brain
            Y, X = np.ogrid[:h, :w]
            cx, cy = w / 2, h / 2
            rx, ry = w * 0.46, h * 0.48
            ellipse_dist = ((X - cx) / rx) ** 2 + ((Y - cy) / ry) ** 2
            # Smooth sigmoid falloff at the ellipse boundary
            weight = 1.0 / (1.0 + np.exp(8.0 * (ellipse_dist - 1.0)))
            weight = weight[:, :, np.newaxis]  # (H,W,1)
            # Blend: inside->original, outside->attenuated (not zeroed)
            blended = arr * (weight + (1 - weight) * (1 - self.blend))
            return Image.fromarray(blended.clip(0, 255).astype(np.uint8))
        except Exception:
            return img  # fallback: return unchanged


class BiasFieldCorrection:
    """
    Approximate bias-field correction using Gaussian smoothing.
    FIXED: sigma=2.0 -> kernel=13x13 (safe for 224x224 images).
    sigma=40 was wrong -- it produced a 241x241 kernel which averaged
    the whole image to one colour, destroying all spatial information.
    sigma=2 correctly estimates local intensity inhomogeneity.
    """
    def __init__(self, sigma=2.0):
        self.sigma = sigma

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB')).astype(np.float32)
            ksize = max(3, int(6 * self.sigma) | 1)  # odd, at least 3
            bias = np.zeros_like(arr)
            for c in range(3):
                bias[..., c] = cv2.GaussianBlur(arr[..., c], (ksize, ksize), self.sigma)
            # Divide by local field estimate (keeps global scale)
            corrected = arr / (bias + 1e-3) * bias.mean(axis=(0, 1), keepdims=True).clip(1e-3)
            return Image.fromarray(corrected.clip(0, 255).astype(np.uint8))
        except Exception:
            return img


class CLAHEEnhancement:
    """
    CLAHE applied to L channel in LAB colour space.
    clip_limit=2.5 (conservative) to avoid over-amplifying noise.
    """
    def __init__(self, clip_limit=2.5, tile_grid=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB'))
            lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
            lab[:, :, 0] = self.clahe.apply(lab[:, :, 0])
            return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))
        except Exception:
            return img


class GaussianDenoising:
    """Mild Gaussian denoising (sigma=0.5). Conservative and safe."""
    def __init__(self, sigma=0.5):
        self.sigma = sigma

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB'))
            return Image.fromarray(cv2.GaussianBlur(arr, (3, 3), self.sigma))
        except Exception:
            return img


class ROIGuidedCrop:
    """
    Gentle hippocampus/medial-temporal-lobe-guided crop.
    FIXED: roi_frac=0.48 (keeps 96% of image on each side).
    Previous value 0.42 cropped too aggressively and, combined with
    skull-strip zeroing + CropBlackBorders, shrank images to tiny patches.
    This version is a subtle refocus, not a destructive crop.
    """
    def __init__(self, cx_frac=0.50, cy_frac=0.53, roi_frac=0.48):
        self.cx = cx_frac; self.cy = cy_frac; self.roi = roi_frac

    def __call__(self, img):
        try:
            w, h = img.size
            hw = int(w * self.roi); hh = int(h * self.roi)
            cx = int(w * self.cx);  cy = int(h * self.cy)
            x0 = max(0, cx - hw); y0 = max(0, cy - hh)
            x1 = min(w, cx + hw); y1 = min(h, cy + hh)
            return img.crop((x0, y0, x1, y1))
        except Exception:
            return img


class CropBlackBorders:
    """Remove black MRI borders."""
    def __init__(self, thr=10, pad=5): self.thr = thr; self.pad = pad
    def __call__(self, img):
        try:
            arr  = np.array(img)
            mask = (arr.sum(axis=2) > self.thr) if arr.ndim == 3 else (arr > self.thr)
            if not mask.any(): return img
            ys, xs = np.where(mask)
            y0 = max(0, ys.min() - self.pad); y1 = min(arr.shape[0]-1, ys.max() + self.pad)
            x0 = max(0, xs.min() - self.pad); x1 = min(arr.shape[1]-1, xs.max() + self.pad)
            return img.crop((x0, y0, x1+1, y1+1))
        except Exception:
            return img


class EnsureRGB:
    def __call__(self, img): return img.convert('RGB')


class ZScorePerImage:
    """Per-image Z-score normalisation -- scanner-invariant."""
    def __call__(self, x):
        m = x.mean(dim=(1, 2), keepdim=True)
        s = x.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
        return (x - m) / s


# ── CORRECT pipeline order ────────────────────────────────────────────────
# 1. EnsureRGB: convert any grayscale to RGB
# 2. CropBlackBorders: remove scanner background BEFORE skull strip
# 3. SkullStripSimulation: SOFT blend (not hard zero), blend_strength=0.1
# 4. BiasFieldCorrection: sigma=2.0 (FIXED from 40.0)
# 5. CLAHEEnhancement: clip_limit=2.5 (conservative)
# 6. GaussianDenoising: sigma=0.5
# 7. ROIGuidedCrop: roi_frac=0.48 (gentle, not destructive)
# 8. Resize to 256x256
# 9. Augmentation (training only)
# 10. ToTensor + ZScorePerImage

train_transform = transforms.Compose([
    EnsureRGB(),
    CropBlackBorders(),                          # remove background first
    SkullStripSimulation(blend_strength=0.1),    # soft blend, not hard zero
    BiasFieldCorrection(sigma=2.0),              # FIXED: sigma=2 not 40
    CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise            # conservative
    GaussianDenoising(sigma=0.5),
    ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),  # gentle
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.5, 1.5))], p=0.3),
    transforms.RandAugment(num_ops=2, magnitude=7),  # auto-selects best augmentations
    transforms.ToTensor(),
    ZScorePerImage(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.10)),  # must be after ToTensor()
])

eval_transform = transforms.Compose([
    EnsureRGB(),
    CropBlackBorders(),
    SkullStripSimulation(blend_strength=0.1),
    BiasFieldCorrection(sigma=2.0),
    CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise
    GaussianDenoising(sigma=0.5),
    ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    ZScorePerImage(),
])


def visualise_preprocessing(df, n_samples=3, seed=42):
    """Show side-by-side preprocessing stages for n sample images."""
    samples = df.sample(min(n_samples, len(df)), random_state=seed)
    stages = [
        ('1.Original',       EnsureRGB()),
        ('2.CropBB',         transforms.Compose([EnsureRGB(), CropBlackBorders()])),
        ('3.SkullStrip(soft)',transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1)])),
        ('4.BiasCorrect',    transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1),
                                                  BiasFieldCorrection(sigma=2.0)])),
        ('5.CLAHE+Denoise',  transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1),
                                                  BiasFieldCorrection(sigma=2.0),
                                                  CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise
                                                  GaussianDenoising(sigma=0.5),
                                                  ROIGuidedCrop(roi_frac=0.48),
                                                  transforms.Resize((IMG_SIZE, IMG_SIZE))])),
    ]
    fig, axes = plt.subplots(len(samples), len(stages),
                             figsize=(4 * len(stages), 3.5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]
    for r, (_, row) in enumerate(samples.iterrows()):
        img_pil = Image.open(row['path'])
        for c, (stage_name, tfm) in enumerate(stages):
            processed = np.array(tfm(img_pil))
            axes[r, c].imshow(processed, cmap='gray' if processed.ndim == 2 else None)
            if r == 0:
                axes[r, c].set_title(stage_name, fontsize=8, fontweight='bold')
            if c == 0:
                axes[r, c].set_ylabel(CLASSES[row['label']], fontsize=7,
                                      rotation=0, labelpad=60, va='center')
            axes[r, c].axis('off')
    plt.suptitle('MRI Preprocessing -- Stage Visualisation (FIXED)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'preprocessing_stages.png',
                dpi=150, bbox_inches='tight')
    plt.close()
    print('[Step 5] Preprocessing visualisation saved.')


try:
    visualise_preprocessing(df_train, n_samples=3)
except NameError:
    print('[Step 5] df_train not yet defined -- run after Section 2.')

print('[Step 5] FIXED preprocessing pipeline configured.')
print('  Key fixes: sigma=2.0 (was 40), soft skull strip (was hard zero),')
print('  roi_frac=0.48 (was 0.42), CropBB BEFORE skull strip.')
print('  Train steps:', [type(t).__name__ for t in train_transform.transforms])


[Step 5] Preprocessing visualisation saved.
[Step 5] FIXED preprocessing pipeline configured.
  Key fixes: sigma=2.0 (was 40), soft skull strip (was hard zero),
  roi_frac=0.48 (was 0.42), CropBB BEFORE skull strip.
  Train steps: ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'RandomResizedCrop', 'RandomHorizontalFlip', 'RandomRotation', 'ColorJitter', 'RandomApply', 'RandAugment', 'ToTensor', 'ZScorePerImage', 'RandomErasing']


## Section 3b -- Multi-Dataset Harmonization & Domain-Shift Evaluation
### Proposal Steps 4 & 27

Implements:
- **ComBat-style** statistical harmonisation (site-mean/std correction)
- **KS test** to quantify pixel-intensity distribution shift between sites
- **MMD** (Maximum Mean Discrepancy) with RBF kernel
- Before/after distribution visualisation
- FedBN keeps site-specific BN statistics local in federated training


In [7]:
# ====================================================================
# STEPS 4 & 27 -- MULTI-DATASET HARMONIZATION & DOMAIN-SHIFT EVALUATION
# ====================================================================

from scipy import stats


def combat_harmonise(df_source, df_target, ref_source='external'):
    """
    ComBat-style harmonisation.
    Shifts pixel intensity statistics of each non-reference site toward
    the reference site mean/std.  Correction factors are stored as
    'site_mean_offset' and 'site_std_scale' columns in the returned df.
    These columns can be used in a custom Dataset to rescale tensors.
    """
    SAMPLE_N = min(200, len(df_source))
    site_stats = {}
    for site in df_source['source'].unique():
        df_s = df_source[df_source['source'] == site].sample(
            min(SAMPLE_N, (df_source['source'] == site).sum()), random_state=42)
        means, stds = [], []
        for _, row in df_s.iterrows():
            try:
                arr = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
                means.append(arr.mean()); stds.append(arr.std() + 1e-6)
            except Exception:
                pass
        if means:
            site_stats[site] = {'mean': float(np.mean(means)), 'std': float(np.mean(stds))}

    if ref_source not in site_stats:
        ref_source = list(site_stats.keys())[0]
    ref_mean = site_stats[ref_source]['mean']
    ref_std  = site_stats[ref_source]['std']

    print(f'[Harmonisation] Reference site: "{ref_source}"  '
          f'mean={ref_mean:.2f}  std={ref_std:.2f}')
    for site, st in site_stats.items():
        offset = ref_mean - st['mean']; scale = ref_std / st['std']
        print(f'  {site:<14}: mean={st["mean"]:.2f}  std={st["std"]:.2f}  '
              f'offset={offset:+.2f}  scale={scale:.3f}')

    df_out = df_target.copy()
    for site, st in site_stats.items():
        mask = df_out['source'] == site
        df_out.loc[mask, 'site_mean_offset'] = ref_mean - st['mean']
        df_out.loc[mask, 'site_std_scale']   = ref_std  / st['std']
    df_out['site_mean_offset'] = df_out.get('site_mean_offset', pd.Series([0.0]*len(df_out))).fillna(0.0)
    df_out['site_std_scale']   = df_out.get('site_std_scale',   pd.Series([1.0]*len(df_out))).fillna(1.0)
    return df_out


def mmd_rbf(X, Y, gamma=1.0):
    """Unbiased MMD squared with RBF kernel. Lower = better alignment."""
    from sklearn.metrics.pairwise import rbf_kernel
    XX = rbf_kernel(X, X, gamma); YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    n, m = len(X), len(Y)
    return float(
        (XX.sum() - np.diag(XX).sum()) / (n * (n - 1)) +
        (YY.sum() - np.diag(YY).sum()) / (m * (m - 1)) -
        2 * XY.mean()
    )


def compute_domain_stats(df, n=300):
    """Sample n images and return per-image mean + std arrays."""
    sample = df.sample(min(n, len(df)), random_state=42)
    means, stds = [], []
    for _, row in sample.iterrows():
        try:
            arr = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
            means.append(arr.mean()); stds.append(arr.std())
        except Exception:
            pass
    return np.array(means), np.array(stds)


try:
    print('[Domain Shift] Computing pixel statistics...')
    means_oasis, stds_oasis = compute_domain_stats(df_oasis)
    means_ext,   stds_ext   = compute_domain_stats(df_external)

    print(f'  OASIS    -- mean: {means_oasis.mean():.2f} +/- {means_oasis.std():.2f}  '
          f'std: {stds_oasis.mean():.2f} +/- {stds_oasis.std():.2f}')
    print(f'  External -- mean: {means_ext.mean():.2f}   +/- {means_ext.std():.2f}  '
          f'std: {stds_ext.mean():.2f}   +/- {stds_ext.std():.2f}')

    ks_stat_m, ks_p_m = stats.ks_2samp(means_oasis, means_ext)
    ks_stat_s, ks_p_s = stats.ks_2samp(stds_oasis,  stds_ext)
    print(f'  KS test (pixel mean): stat={ks_stat_m:.4f}  p={ks_p_m:.4f}  '
          f'{"** SIGNIFICANT **" if ks_p_m < 0.05 else "No significant shift"}')
    print(f'  KS test (pixel std) : stat={ks_stat_s:.4f}  p={ks_p_s:.4f}  '
          f'{"** SIGNIFICANT **" if ks_p_s < 0.05 else "No significant shift"}')

    X_oasis = np.stack([means_oasis, stds_oasis], axis=1)
    X_ext   = np.stack([means_ext,   stds_ext],   axis=1)
    mmd_val = mmd_rbf(X_oasis, X_ext, gamma=1.0 / (X_oasis.var() + 1e-6))
    print(f'  MMD2 (RBF)          : {mmd_val:.6f}  '
          f'{"high shift" if mmd_val > 0.01 else "low shift"}')

    # Visualise domain shift
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(means_oasis, bins=40, alpha=0.7, label='OASIS',    color='#3498db', density=True)
    axes[0].hist(means_ext,   bins=40, alpha=0.7, label='External', color='#e74c3c', density=True)
    axes[0].set(title=f'Pixel Mean (KS p={ks_p_m:.4f})',
                xlabel='Mean Intensity', ylabel='Density')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].hist(stds_oasis, bins=40, alpha=0.7, label='OASIS',    color='#3498db', density=True)
    axes[1].hist(stds_ext,   bins=40, alpha=0.7, label='External', color='#e74c3c', density=True)
    axes[1].set(title=f'Pixel Std (KS p={ks_p_s:.4f})',
                xlabel='Std Intensity', ylabel='Density')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('Domain Shift Evaluation: OASIS vs External Dataset', fontsize=12)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'domain_shift.png', dpi=150)
    plt.close()
    print('[Step 4/27] Domain-shift plot saved.')

    # Apply ComBat harmonisation
    print('[Harmonisation] Applying ComBat-style correction...')
    df_train         = combat_harmonise(df_external, df_train)
    df_val           = combat_harmonise(df_external, df_val)
    df_test_internal = combat_harmonise(df_external, df_test_internal)
    df_test_external = combat_harmonise(df_external, df_test_external)

    harm_summary = {
        'mmd_before': mmd_val, 'ks_mean_stat': float(ks_stat_m),
        'ks_mean_p': float(ks_p_m), 'ks_std_stat': float(ks_stat_s),
        'ks_std_p': float(ks_p_s),
        'oasis_pixel_mean': float(means_oasis.mean()),
        'external_pixel_mean': float(means_ext.mean()),
    }
    import json as _json
    with open(SAVE_DIR / 'results' / 'harmonisation_summary.json', 'w') as _f:
        _json.dump(harm_summary, _f, indent=2)
    print('[Step 4/27] Harmonisation complete.\n')

except NameError as e:
    print(f'[Step 4/27] DataFrames not yet defined: {e}. Run after Section 2.')


[Domain Shift] Computing pixel statistics...
  OASIS    -- mean: 41.14 +/- 6.60  std: 44.47 +/- 3.96
  External -- mean: 72.17   +/- 6.97  std: 83.41   +/- 6.94
  KS test (pixel mean): stat=0.9667  p=0.0000  ** SIGNIFICANT **
  KS test (pixel std) : stat=0.9967  p=0.0000  ** SIGNIFICANT **
  MMD2 (RBF)          : 0.534371  high shift
[Step 4/27] Domain-shift plot saved.
[Harmonisation] Applying ComBat-style correction...
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Step 4/27] Harmonisation complete.



In [8]:
# Helper classes kept for backward compatibility (used in cell 10)
# train_transform and eval_transform are defined in Section 3a (cell 10)
# with full deep preprocessing pipeline. Do NOT redefine them here.

class EnsureRGB:
    def __call__(self, img): return img.convert('RGB')

class CropBlackBorders:
    """Remove black MRI borders by finding the bounding box of non-zero pixels."""
    def __init__(self, thr=10, pad=5): self.thr = thr; self.pad = pad
    def __call__(self, img):
        arr  = np.array(img)
        mask = (arr.sum(axis=2) > self.thr) if arr.ndim == 3 else (arr > self.thr)
        if not mask.any(): return img
        ys, xs = np.where(mask)
        y0 = max(0, ys.min() - self.pad); y1 = min(arr.shape[0]-1, ys.max() + self.pad)
        x0 = max(0, xs.min() - self.pad); x1 = min(arr.shape[1]-1, xs.max() + self.pad)
        return img.crop((x0, y0, x1+1, y1+1))

class ZScorePerImage:
    """Per-image Z-score normalisation -- scanner-invariant."""
    def __call__(self, x):
        m = x.mean(dim=(1, 2), keepdim=True)
        s = x.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
        return (x - m) / s

# Confirm transforms from cell 10 are active
print('Helper classes ready. train_transform / eval_transform from Section 3a are in effect.')
print('  Train:', [type(t).__name__ for t in train_transform.transforms])
print('  Eval :', [type(t).__name__ for t in eval_transform.transforms])


Helper classes ready. train_transform / eval_transform from Section 3a are in effect.
  Train: ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'RandomResizedCrop', 'RandomHorizontalFlip', 'RandomRotation', 'ColorJitter', 'RandomApply', 'RandAugment', 'ToTensor', 'ZScorePerImage', 'RandomErasing']
  Eval : ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'CenterCrop', 'ToTensor', 'ZScorePerImage']


## Section 4 — Dataset Class & DataLoaders

In [9]:
class AlzheimerDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df=df.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        with Image.open(row["path"]) as img:
            img = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return img, int(row["label"]), row["path"]

def make_loader(df_part, transform, shuffle, bs=BATCH_SIZE):
    ds = AlzheimerDataset(df_part, transform)
    if shuffle:  # WeightedRandomSampler — fixes 137:1 class imbalance
        labels  = df_part["label"].values
        counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
        counts  = np.maximum(counts, 1)        # avoid div by zero
        weights = 1.0 / counts[labels]         # inverse-frequency weights
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.DoubleTensor(weights),
            num_samples=len(weights), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS>0),
                          drop_last=True)   # FIX: never yield a final batch of
                                            # size 1 -- BatchNorm1d in classifier
                                            # crashes in .train() mode on batch=1
    return DataLoader(ds, batch_size=bs, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS>0))

def class_weights(df):
    counts = (df["label"].value_counts().sort_index()
                .reindex(range(NUM_CLASSES), fill_value=0).values)
    counts = np.maximum(counts, 1).astype(np.float32)
    w = counts.sum() / (NUM_CLASSES * counts)
    return torch.tensor(w, device=device)

print("Dataset class ready.")


Dataset class ready.


## Section 5 — Model Architecture
### 5a CBAM

In [10]:
# ================================================================
# SECTION 5 -- MODEL ARCHITECTURE (COMPLETE)
# Dual-Channel ConvNeXt-Small + Swin-V2-S + CBAM
# Fixes: missing backbones, forward_features, Dropout, drop_path
# ================================================================

import timm
import torch.nn as nn
import torch

# ── 5a. CBAM (Channel + Spatial Attention) ──────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        r = max(1, channels // reduction)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(r, channels, bias=False),
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(
            self.fc(self.avg(x)) + self.fc(self.max(x))
        ).unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        return x * self.sig(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_k=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_k)
    def forward(self, x):
        return self.sa(self.ca(x))


# ── 5b. Dual-Channel Model: ConvNeXt-Tiny + Swin-V2-T + CBAM ───────────
class DualChannelAD(nn.Module):
    """
    Dual-channel feature fusion:
      Channel A: ConvNeXt-Small (local texture + CNN inductive bias)
      Channel B: Swin-V2-S     (global context + hierarchical attention)
    Features fused via concatenation, refined by CBAM, classified by MLP head.
    forward_features() exposed for contrastive loss.
    Stochastic depth (drop_path) applied inside timm backbones automatically.
    """
    def __init__(self, num_classes=4, bottleneck_dim=512,
                 dropout=0.4, freeze_ratio=0.5):
        super().__init__()
        # Channel A: ConvNeXt-Tiny
        self.convnext = timm.create_model(
            'convnext_small', pretrained=True, num_classes=0,
            drop_path_rate=0.2)        # ↑ tiny→small: +22M params, deeper net        # stochastic depth
        feat_a = self.convnext.num_features  # 768

        # Channel B: Swin-V2-T
        self.swin = timm.create_model(
            'swinv2_small_window8_256', pretrained=True, num_classes=0,
            drop_path_rate=0.2,   # ↑ tiny→small: +22M params
            img_size=IMG_SIZE)         # stochastic depth
        feat_b = self.swin.num_features      # 768

        feat_total = feat_a + feat_b         # 1536

        # Freeze lower layers of backbones (transfer learning)
        self._freeze_backbones(freeze_ratio)

        # Reshape fusion features for CBAM (treat as 1D 'spatial' of 1x1)
        self.cbam = CBAM(channels=feat_total, reduction=16)
        # Project fused features to bottleneck
        self.projector = nn.Sequential(
            nn.LayerNorm(feat_total),
            nn.Linear(feat_total, bottleneck_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(bottleneck_dim),  # stabilizes final features
            nn.Linear(bottleneck_dim, bottleneck_dim // 2),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(bottleneck_dim // 2, num_classes),
        )

    def _freeze_backbones(self, ratio):
        """Freeze the first `ratio` fraction of layers in each backbone."""
        for backbone in [self.convnext, self.swin]:
            params = list(backbone.parameters())
            n_freeze = int(len(params) * ratio)
            for p in params[:n_freeze]:
                p.requires_grad = False

    def forward_features(self, x):
        """Returns bottleneck embedding (used by contrastive loss)."""
        # Extract features from both channels
        fa = self.convnext(x)           # (B, 768)
        fb = self.swin(x)               # (B, 768)

        # Fuse
        fused = torch.cat([fa, fb], dim=1)  # (B, 1536)

        # CBAM on fused (unsqueeze to (B, C, 1, 1) for spatial ops)
        fused_4d  = fused.unsqueeze(-1).unsqueeze(-1)  # (B, 1536, 1, 1)
        fused_att = self.cbam(fused_4d).squeeze(-1).squeeze(-1)  # (B, 1536)

        # Bottleneck projection
        return self.projector(fused_att)   # (B, bottleneck_dim)

    def forward(self, x):
        feats  = self.forward_features(x)
        return self.classifier(feats)


# Alias used throughout the notebook
DualChannelModel = DualChannelAD

# Quick sanity check
try:
    _m   = DualChannelAD(NUM_CLASSES, 512, 0.4, 0.5).to(device)
    _x   = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _f   = _m.forward_features(_x)
    _out = _m(_x)
    _tp  = sum(p.numel() for p in _m.parameters()) / 1e6
    _tr  = sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6
    print(f'[Model OK] DualChannelAD')
    print(f'  Input    : {list(_x.shape)}')
    print(f'  Features : {list(_f.shape)}')
    print(f'  Output   : {list(_out.shape)}')
    print(f'  Params   : {_tp:.1f}M total | {_tr:.1f}M trainable')
    del _m, _x, _f, _out
    torch.cuda.empty_cache()
except Exception as _e:
    print(f'[Model check failed] {_e}')
    print('  swinv2_small_window8_256 + IMG_SIZE=256: all feature maps divisible by 8')
    print('  IMG_SIZE=256, swinv2_small_window8_256: window8 divides all feature maps perfectly')
print('Model architecture ready.')


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

[Model OK] DualChannelAD
  Input    : [2, 3, 256, 256]
  Features : [2, 512]
  Output   : [2, 4]
  Params   : 99.6M total | 68.5M trainable
Model architecture ready.


In [11]:
# ====================================================================
# CHECKPOINT RESUME LOADER
# Loads rescued checkpoints to safely skip processing & evaluation.
# ====================================================================
import json, pickle
from pathlib import Path

RESCUED_DIR = Path('/kaggle/input/datasets/claudeagent/model-3/model3')

# 1. Load DataFrames
data_ckpt = RESCUED_DIR / 'ckpt_data.pkl'
if data_ckpt.exists():
    with open(data_ckpt, 'rb') as f:
        _d = pickle.load(f)
    df_oasis = _d['df_oasis']
    df_external = _d['df_external']
    df_train = _d['df_train']
    df_val = _d['df_val']
    df_test_internal = _d['df_test_internal']
    df_test_external = _d['df_test_external']
    print(f'[LOADED] DataFrames from {data_ckpt}')
    print(f'  train={len(df_train)} | internal={len(df_test_internal)} | external={len(df_test_external)}')
else:
    print('[WARN] ckpt_data.pkl not found! You MUST run Section 2.')

# 2. Load Evaluation Metrics
eval_ckpt = RESCUED_DIR / 'ckpt_all_evals.json'
if eval_ckpt.exists():
    with open(eval_ckpt, 'r') as f:
        all_evals = json.load(f)
    print(f'\n[LOADED] all_evals from {eval_ckpt}')
    print(f'  Found {len(all_evals)} evaluated models.')
else:
    print('[WARN] ckpt_all_evals.json not found! You MUST run Section 11.')


[LOADED] DataFrames from /kaggle/input/datasets/claudeagent/model-3/model3/ckpt_data.pkl
  train=4489 | internal=951 | external=86437

[LOADED] all_evals from /kaggle/input/datasets/claudeagent/model-3/model3/ckpt_all_evals.json
  Found 8 evaluated models.


### Load Part 1 + Part 2 Checkpoints

Loads centralized model from Part 1 and all FL models from Part 2.

In [12]:
# ====================================================================
# LOAD FROM PART 1 + PART 2
# Add alzheimer-part1-output AND alzheimer-part2-output as Kaggle inputs
# ====================================================================
import json, torch, pickle
from pathlib import Path

P1_DIR = Path('/kaggle/input/datasets/claudeagent/model-1')
P2_DIR = Path('/kaggle/input/datasets/claudeagent/model-2/models2')
if not P1_DIR.exists(): P1_DIR = SAVE_DIR
if not P2_DIR.exists(): P2_DIR = SAVE_DIR
print(f"[Load] Part1 dir: {P1_DIR}")
print(f"[Load] Part2 dir: {P2_DIR}")

# ── Load PARAMS ──
with open(P1_DIR / 'best_params.json') as f:
    PARAMS = json.load(f)
print(f"[Load] PARAMS: {PARAMS}")

# ── Load centralized model ──
cent_model = DualChannelAD(
    NUM_CLASSES, PARAMS['bd'], PARAMS['dr'], PARAMS['fr']
).to(device)
cent_model.load_state_dict(
    torch.load(P1_DIR / 'best_fold_model.pt', map_location=device)
)
cent_model.eval()
print("[Load] cent_model ready.")

# ── Load fold histories (for training curve plots) ──
fold_hist_path = P1_DIR / 'fold_histories.pkl'
if fold_hist_path.exists():
    with open(fold_hist_path, 'rb') as f:
        fold_histories = pickle.load(f)
    print("[Load] fold_histories loaded.")
else:
    fold_histories = {}
    print("[Load] fold_histories not found — curve plots may be skipped.")

# ── Load FL models ──
fl_models = {}
fl_hists  = {}
# Loads: fedavg_final.pt, fedprox_final.pt, fedbn_final.pt
for algo in ['FedAvg', 'FedProx', 'FedBN']:
    pt_path = P2_DIR / f'{algo.lower()}_final.pt'  # fedavg_final.pt etc.
    if pt_path.exists():
        m = DualChannelAD(
            NUM_CLASSES, PARAMS['bd'], PARAMS['dr'], PARAMS['fr']
        ).to(device)
        m.load_state_dict(torch.load(pt_path, map_location=device))
        m.eval()
        fl_models[algo] = m
        print(f"[Load] {algo} model loaded.")
    else:
        print(f"[WARN] {algo} model not found at {pt_path} — skipping.")

fl_hist_path = P2_DIR / 'fl_histories.pkl'
if fl_hist_path.exists():
    with open(fl_hist_path, 'rb') as f:
        fl_hists = pickle.load(f)
    print("[Load] fl_histories loaded.")

print()
print(f"[Load] Ready: cent_model + FL models: {list(fl_models.keys())}")
print("[Load] Starting evaluation...")


[Load] Part1 dir: /kaggle/input/datasets/claudeagent/model-1
[Load] Part2 dir: /kaggle/input/datasets/claudeagent/model-2/models2
[Load] PARAMS: {'bd': 512, 'fr': 0.5394633936788146, 'dr': 0.36240745617697456, 'lr': 6.612372870684137e-05, 'wd': 0.00012551115172973836, 'ep': 24, 'l1': 0.4404460046972835, 'l2': 0.4832290311184182, 'l3': 0.07632496418429824, 'pat': 8}
[Load] cent_model ready.
[Load] fold_histories not found — curve plots may be skipped.
[Load] FedAvg model loaded.
[Load] FedProx model loaded.
[Load] FedBN model loaded.
[Load] fl_histories loaded.

[Load] Ready: cent_model + FL models: ['FedAvg', 'FedProx', 'FedBN']
[Load] Starting evaluation...


### 5b Dual-Channel ConvNeXt-Tiny + Swin-V2-T Model

In [13]:
# Cell originally contained old DualChannelAD (OLD-REMOVED).
# REMOVED: superseded by the complete DualChannelAD in Section 5 (Cell 21).
# DualChannelAD = the class defined in Cell 21 is already active.
print('DualChannelAD: using upgraded version from Section 5 (Cell 21). OK.')


DualChannelAD: using upgraded version from Section 5 (Cell 21). OK.


## Section 6 — Hybrid Loss Function

In [14]:
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).
    Down-weights easy examples, focuses on hard/minority cases.
    gamma=2.0 is standard. Used as one component of HybridLoss.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, y):
        ce  = F.cross_entropy(logits, y, weight=self.weight, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# ── Supervised Contrastive Loss ──────────────────────────────────────
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    Pulls same-class embeddings together, pushes different-class apart.
    Significantly reduces inter-class confusion especially for adjacent
    AD stages (VeryMild <-> Mild).
    temperature=0.07 is the standard setting from the paper.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """features: (B, D) L2-normalised embeddings. labels: (B,) int."""
        device = features.device
        features = F.normalize(features, dim=1)
        B = features.shape[0]
        if B < 2:
            return torch.tensor(0.0, device=device)
        sim   = torch.matmul(features, features.T) / self.temperature
        # Mask diagonal (self-similarity)
        mask_eye = torch.eye(B, dtype=torch.bool, device=device)
        sim.masked_fill_(mask_eye, float('-inf'))
        # Positive mask: same class
        labels   = labels.view(-1, 1)
        pos_mask = (labels == labels.T).float()
        pos_mask.fill_diagonal_(0)
        n_pos    = pos_mask.sum(dim=1).clamp(min=1)
        log_prob = F.log_softmax(sim, dim=1)
        loss     = -(pos_mask * log_prob).sum(dim=1) / n_pos
        loss = loss.clamp(min=-50, max=50)  # prevent inf
        loss_val = loss.mean()
        if torch.isnan(loss_val) or torch.isinf(loss_val):
            loss_val = torch.tensor(0.0, requires_grad=True, device=loss.device)
        return loss_val


# ── Upgraded HybridLoss with Soft-Target support for Mixup ───────────
class HybridLoss(nn.Module):
    """
    Hybrid loss = lam1*CE + lam2*Focal + lam3*SupCon
    Upgraded to accept y_soft (soft labels from Mixup).
    When y_soft is provided, CE and Focal use KL-div against soft targets.
    """
    def __init__(self, class_weights=None, lam1=0.4, lam2=0.4, lam3=0.2,
                 gamma=2.0, label_smoothing=LABEL_SMOOTHING):  # uses config var (currently 0.1)
        super().__init__()
        self.lam1 = lam1; self.lam2 = lam2; self.lam3 = lam3
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=label_smoothing)
        self.supcon = SupConLoss(temperature=0.07)

    def focal(self, logits, y_soft):
        """Focal loss with soft targets."""
        p      = torch.softmax(logits, dim=1)
        log_p  = torch.log_softmax(logits, dim=1)
        weight = (1 - p) ** self.gamma
        return -(weight * log_p * y_soft).sum(dim=1).mean()

    def forward(self, logits, features, labels, y_soft=None):
        """
        logits   : (B, C) raw model output
        features : (B, D) bottleneck embedding for SupCon
        labels   : (B,)   hard integer labels
        y_soft   : (B, C) soft labels from Mixup (optional)
        """
        if y_soft is None:
            y_soft = F.one_hot(labels, logits.size(1)).float()
        # CE with soft targets via KL-div
        log_p   = F.log_softmax(logits, dim=1)
        ce_loss = -(y_soft * log_p).sum(dim=1).mean()
        # Focal with soft targets
        fc_loss = self.focal(logits, y_soft)
        # SupCon uses hard labels (Mixup labels don't affect contrastive)
        sc_loss = self.supcon(features, labels)
        # NaN guards -- prevent any single loss from poisoning training
        if torch.isnan(ce_loss): ce_loss = torch.tensor(0.0, device=ce_loss.device)
        if torch.isnan(fc_loss): fc_loss = torch.tensor(0.0, device=fc_loss.device)
        if torch.isnan(sc_loss): sc_loss = torch.tensor(0.0, device=sc_loss.device)
        return self.lam1 * ce_loss + self.lam2 * fc_loss + self.lam3 * sc_loss

print('HybridLoss ready (CE + Focal + SupCon + soft-target Mixup support).')


HybridLoss ready (CE + Focal + SupCon + soft-target Mixup support).


## Section 7 — Training Utilities

In [15]:

def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation — stronger than Mixup for medical images."""
    if alpha <= 0 or np.random.rand() > 0.5:
        return x, y, y, 1.0  # skip 50% of the time
    lam  = np.random.beta(alpha, alpha)
    B    = x.size(0)
    idx  = torch.randperm(B, device=x.device)
    W, H = x.size(3), x.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx    = np.random.randint(W)
    cy    = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x_mix, y, y[idx], lam

# ====================================================================
# SECTION 7 -- UPGRADED TRAINING UTILITIES
# Adds: EMA, Cosine Warmup LR, Mixup, Label Smoothing, Grad Clip
# These are the primary drivers of smooth curves and ~96% accuracy.
# ====================================================================

# ── Exponential Moving Average (EMA) ─────────────────────────────────────
class EMA:
    """
    Maintains a shadow copy of model weights as an exponential moving average.
    EMA weights produce smoother, more stable predictions.
    Apply EMA weights at validation/test time for best accuracy.
    """
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = {k: v.clone().float() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.float()

    def apply(self, model):
        """Load EMA weights into model for inference."""
        model.load_state_dict({k: v.to(next(model.parameters()).device)
                               for k, v in self.shadow.items()})

    def state_dict(self):
        return self.shadow


# ── Cosine Annealing with Linear Warmup ──────────────────────────────────
class WarmupCosineScheduler:
    """
    Linear warmup for warmup_epochs, then cosine annealing to min_lr.
    Warmup prevents instability in early training (large pretrained models).
    Cosine decay produces smooth, stable convergence without oscillation.
    """
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 base_lr, min_lr=1e-6):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr       = base_lr
        self.min_lr        = min_lr
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        ep = self.current_epoch
        if ep <= self.warmup_epochs:
            lr = self.base_lr * ep / self.warmup_epochs
        else:
            import math
            progress = (ep - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
                1.0 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']


# ── Mixup augmentation ────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.2, num_classes=4):
    """
    Mixup: interpolate pairs of training samples.
    Produces smoother decision boundaries and reduces overconfidence.
    Returns mixed inputs and soft targets as a float tensor.
    """
    import numpy as np
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch = x.size(0)
    idx   = torch.randperm(batch, device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a   = F.one_hot(y, num_classes).float()
    y_b   = F.one_hot(y[idx], num_classes).float()
    y_mix = lam * y_a + (1 - lam) * y_b
    return x_mix, y_mix


# ── Upgraded train_one_epoch with Mixup + Grad Clip + EMA ────────────────
# Mixed-precision scaler (AMP) — ~2x faster training, no accuracy loss
_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def train_one_epoch(model, loader, optimizer, criterion,
                    ema=None, mixup_alpha=0.2, grad_clip=1.0,
                    scheduler=None):
    model.train()
    total_loss = 0.0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)

        # Mixup (training only)
        if mixup_alpha > 0:
            x_in, y_soft = mixup_batch(x, y, mixup_alpha, NUM_CLASSES)
        else:
            x_in  = x
            y_soft = F.one_hot(y, NUM_CLASSES).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            feats  = model.forward_features(x_in)
            logits = model.classifier(feats)
            # Criterion accepts soft targets when mixup is active
            loss = criterion(logits, feats, y, y_soft=y_soft)

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(); continue  # skip bad batch
        
        _scaler.scale(loss).backward()
        _scaler.unscale_(optimizer)
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        _scaler.step(optimizer)
        _scaler.update()

        # Update EMA after each step
        if ema is not None:
            ema.update(model)

        total_loss += loss.item() * x.size(0)

    # Step warmup/cosine scheduler per epoch
    if scheduler is not None and isinstance(scheduler, WarmupCosineScheduler):
        scheduler.step()

    return total_loss / len(loader.dataset)


# ── Eval epoch (unchanged but EMA-aware) ─────────────────────────────────
@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    probs_all, preds_all, lbls_all = [], [], []
    for x, y, _ in loader:
        logits = model(x.to(device))
        p = torch.softmax(logits, 1).cpu().numpy()
        probs_all.append(p)
        preds_all.append(p.argmax(1))
        lbls_all.append(y.numpy())
    return (np.concatenate(lbls_all),
            np.concatenate(preds_all),
            np.concatenate(probs_all))


# ── Metrics ────────────────────────────────────────────────────────────────
def all_metrics(lbls, preds, probs):
    m = {
        'accuracy':        accuracy_score(lbls, preds),
        'balanced_acc':    balanced_accuracy_score(lbls, preds),
        'f1_macro':        f1_score(lbls, preds, average='macro',    zero_division=0),
        'f1_weighted':     f1_score(lbls, preds, average='weighted', zero_division=0),
        'f2_macro':        fbeta_score(lbls, preds, beta=2, average='macro', zero_division=0),
        'precision_macro': precision_score(lbls, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(lbls, preds, average='macro',    zero_division=0),
        'mcc':             matthews_corrcoef(lbls, preds),
    }
    try:
        try:
            m['auc_macro']    = roc_auc_score(lbls, probs, multi_class='ovr', average='macro')
            m['auc_weighted'] = roc_auc_score(lbls, probs, multi_class='ovr', average='weighted')
        except Exception:
            # Fallback: model predicting single class (NaN loss collapse)
            m['auc_macro'] = m['auc_weighted'] = 0.5
    except Exception:
        m['auc_macro'] = m['auc_weighted'] = float('nan')
    return m


# ── Smooth curve plotting helper ──────────────────────────────────────────
def smooth(values, weight=0.85):
    """
    Exponential moving average smoothing for training curves.
    weight=0.85 gives visually smooth curves without losing trend information.
    """
    smoothed, last = [], values[0]
    for v in values:
        last = last * weight + v * (1 - weight)
        smoothed.append(last)
    return smoothed


def plot_training_curves(fold_histories, save_path=None):
    """
    Plot smooth training curves for all folds:
    - Loss (train)
    - Accuracy (val)
    - AUC-macro (val)
    - F1-macro (val)
    - Learning Rate
    """
    metrics = [
        ('loss',        'Training Loss',        '#e74c3c', True),
        ('accuracy',    'Val Accuracy',         '#3498db', False),
        ('auc_macro',   'Val AUC-macro',        '#2ecc71', False),
        ('f1_macro',    'Val F1-macro',         '#9b59b6', False),
    ]
    n_folds = len(fold_histories)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()

    colors = plt.cm.tab10(np.linspace(0, 1, n_folds))

    for ax_i, (metric, title, base_color, is_loss) in enumerate(metrics):
        ax = axes[ax_i]
        all_vals = []
        for fold_i, hist in enumerate(fold_histories):
            vals = [h.get(metric, float('nan')) for h in hist]
            eps  = [h['ep'] for h in hist]
            s_vals = smooth(vals, weight=0.85)
            # Raw (faint)
            ax.plot(eps, vals, alpha=0.18, color=colors[fold_i], linewidth=1)
            # Smoothed (bold)
            ax.plot(eps, s_vals, alpha=0.9,
                    color=colors[fold_i], linewidth=2.0,
                    label=f'Fold {fold_i+1}')
            all_vals.extend(vals)

        # Mean across folds
        max_ep = max(len(h) for h in fold_histories)
        mean_curve = []
        for ep in range(max_ep):
            ep_vals = [hist[ep].get(metric, float('nan'))
                       for hist in fold_histories if ep < len(hist)]
            mean_curve.append(np.nanmean(ep_vals))
        s_mean = smooth(mean_curve, weight=0.85)
        ax.plot(range(1, len(s_mean)+1), s_mean,
                color='black', linewidth=2.8, linestyle='--',
                label='Mean (smooth)', zorder=10)

        # Best value annotation
        best_val = min(all_vals) if is_loss else max(all_vals)
        ax.axhline(best_val, color='gray', linestyle=':', alpha=0.5,
                   linewidth=1.2)
        ax.text(0.98, best_val, f' Best: {best_val:.4f}',
                transform=ax.get_yaxis_transform(),
                ha='right', va='bottom', fontsize=8, color='gray')

        ax.set(title=title, xlabel='Epoch', ylabel=metric)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Training Curves -- Smooth (EMA) + Raw (faint)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = save_path or (SAVE_DIR / 'figures' / 'training_curves_smooth.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[Curves] Smooth training curves saved -> {out}')


# ── Full training loop with EMA + Cosine Warmup (replaces train_es) ───────
def train_es(model, dl_tr, dl_va, optimizer, criterion,
             scheduler=None, max_ep=None, patience=None):
    """
    Upgraded training loop:
    - EMA weights tracked throughout
    - WarmupCosine scheduler used if scheduler is WarmupCosineScheduler
    - Mixup applied every batch
    - Gradient clipping every step
    - Best model selected by EMA-evaluated AUC
    - Smooth curves saved automatically
    """
    max_ep   = max_ep  or MAX_EPOCHS
    patience = patience or PATIENCE

    ema        = EMA(model, decay=EMA_DECAY)
    best_auc   = -1.0
    best_state = None
    best_ep    = 0
    bad        = 0
    hist       = []

    # Store original weights to restore after EMA eval
    for ep in range(1, max_ep + 1):
        # Progressive unfreezing: unfreeze all at epoch 10
        if ep == 10:
            for param in model.parameters():
                param.requires_grad = True
        loss = train_one_epoch(
            model, dl_tr, optimizer, criterion,
            ema=ema, mixup_alpha=MIXUP_ALPHA, grad_clip=GRAD_CLIP,
            scheduler=scheduler if isinstance(scheduler, WarmupCosineScheduler) else None
        )

        # Evaluate with EMA weights
        _orig_state = copy.deepcopy(model.state_dict())
        ema.apply(model)
        lbls, preds, probs = eval_epoch(model, dl_va)
        model.load_state_dict(_orig_state)  # restore training weights

        m   = all_metrics(lbls, preds, probs)
        auc_val = m['auc_macro']
        hist.append({'ep': ep, 'loss': loss, **m})

        # Step ReduceLROnPlateau if used instead of cosine
        if scheduler is not None and isinstance(scheduler,
                torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(auc_val)

        cur_lr = optimizer.param_groups[0]['lr']

        print(f'Ep{ep:03d} loss={loss:.4f} AUC={auc_val:.4f} '
              f'Acc={m["accuracy"]:.4f} F1={m["f1_macro"]:.4f} '
              f'LR={cur_lr:.2e} '
              f'best={best_auc:.4f}@{best_ep} pat={bad}/{patience}')

        if auc_val > best_auc + 1e-4:
            best_auc   = auc_val
            best_ep    = ep
            bad        = 0
            # Save EMA weights as best state
            best_state = {k: v.cpu().clone() for k, v in ema.state_dict().items()}
        else:
            bad += 1

        if bad >= patience:
            print(f'Early stop @ ep{ep}  best AUC={best_auc:.4f}@ep{best_ep}')
            break

    # Load best EMA state into model
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return {
        'best_state': best_state,
        'best_ep':    best_ep,
        'best_auc':   best_auc,
        'hist':       hist,
    }


print('Training utilities ready (EMA + WarmupCosine + Mixup + GradClip + SmoothCurves).')

# ── Test Time Augmentation (TTA) ─────────────────────────────────────────────
# Runs inference with N augmented views and averages predictions.
# Adds ~1-2% accuracy at test time with zero extra training.
def tta_predict(model, df, n_aug=5, batch_size=BATCH_SIZE):
    """TTA: average predictions over n_aug augmented views."""
    import torch.nn.functional as F
    tta_transform = transforms.Compose([
        EnsureRGB(), CropBlackBorders(),
        SkullStripSimulation(blend_strength=0.1),
        BiasFieldCorrection(sigma=2.0),
        CLAHEEnhancement(clip_limit=1.5),
        GaussianDenoising(sigma=0.5),
        ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(), ZScorePerImage(),
    ])
    model.eval()
    all_probs = []
    with torch.no_grad():
        for _ in range(n_aug):
            ds  = MRIDataset(df, tta_transform)
            dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
            probs_run = []
            for x, _, _ in dl:
                x = x.to(device)
                out = model(x)
                probs_run.append(F.softmax(out, dim=1).cpu().numpy())
            all_probs.append(np.concatenate(probs_run, axis=0))
    return np.mean(all_probs, axis=0)  # average across n_aug views
print("TTA function ready.")


# ── Model Ensemble ────────────────────────────────────────────────────────────
# Average predictions from all fold models for maximum accuracy.
def ensemble_predict(models, df, use_tta=False, n_aug=3):
    """Ensemble: average softmax predictions from multiple models."""
    import torch.nn.functional as F
    all_model_probs = []
    for mdl in models:
        mdl.eval()
        if use_tta:
            probs = tta_predict(mdl, df, n_aug=n_aug)
        else:
            dl = make_loader(df, eval_transform, False)
            run_probs = []
            with torch.no_grad():
                for x, _, _ in dl:
                    out = mdl(x.to(device))
                    run_probs.append(F.softmax(out, dim=1).cpu().numpy())
            probs = np.concatenate(run_probs, axis=0)
        all_model_probs.append(probs)
    return np.mean(all_model_probs, axis=0)  # average across models
print("Ensemble function ready.")


Training utilities ready (EMA + WarmupCosine + Mixup + GradClip + SmoothCurves).
TTA function ready.
Ensemble function ready.


# DISABLED: SKIPPING HEAVY EVALUATION
if False:
    ## Section 11 — Full Evaluation + Figures

In [16]:
def plot_cm(lbls, preds, title, path):
    cm  = confusion_matrix(lbls,preds)
    cmn = cm.astype(float)/cm.sum(1,keepdims=True)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for ax,data,fmt,t in zip(axes,[cm,cmn],["d",".2f"],["Counts","Normalised"]):
        sns.heatmap(data,annot=True,fmt=fmt,ax=ax,cmap="Blues",
                    xticklabels=CLASSES,yticklabels=CLASSES)
        ax.set(title=f"{title} {t}",xlabel="Predicted",ylabel="True")
    plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()

def plot_roc(lbls, probs, title, path):
    fig,ax=plt.subplots(figsize=(8,6))
    colors=["#e74c3c","#3498db","#2ecc71","#9b59b6"]
    for i,(cls,col) in enumerate(zip(CLASSES,colors)):
        fpr,tpr,_=roc_curve((lbls==i).astype(int),probs[:,i])
        ax.plot(fpr,tpr,color=col,lw=2,label=f"{cls} AUC={auc(fpr,tpr):.3f}")
    ax.plot([0,1],[0,1],"k--",lw=1)
    ax.set(xlabel="FPR",ylabel="TPR",title=f"Per-Class ROC — {title}")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()

# Best centralised model — already loaded from Part 1 output
# cent_model was loaded in the Load section above (best_fold_model.pt)
# Just save a copy to our output folder
torch.save(cent_model.state_dict(), SAVE_DIR/"models"/"centralized_best.pt")
print("[Eval] Centralized model ready from Part 1 checkpoint.")

eval_sets  = {"Internal_Test":df_test_internal, "External_Test":df_test_external}
all_evals  = {}
for mname,mdl in {"Centralized":cent_model, **fl_models}.items():
    mdl.eval()
    for sname,df_s in eval_sets.items():
        lbls,preds,probs = eval_epoch(mdl, make_loader(df_s,eval_transform,False))
        m = all_metrics(lbls,preds,probs); key=f"{mname}_{sname}"
        all_evals[key] = m
        plot_cm(lbls,preds,key,  SAVE_DIR/"figures"/f"cm_{key}.png")
        plot_roc(lbls,probs,key, SAVE_DIR/"figures"/f"roc_{key}.png")
        print(f"[{key}] Acc={m['accuracy']:.4f} AUC={m['auc_macro']:.4f} F1={m['f1_macro']:.4f}")

pd.DataFrame(all_evals).T.to_csv(SAVE_DIR/"results"/"eval_results.csv")
print("Evaluation complete.")

# ── Per-class report + balanced accuracy ─────────────────────────────
def detailed_eval_report(model, df, split_name, model_name='Model'):
    """
    Full evaluation: AUC, F1, accuracy, balanced-accuracy,
    per-class precision/recall/F1, and classification report.
    """
    lbls, preds, probs = eval_epoch(
        model, make_loader(df, eval_transform, False))
    m = all_metrics(lbls, preds, probs)
    m['balanced_acc'] = balanced_accuracy_score(lbls, preds)

    print(f'\n{"-"*60}')
    print(f'  {model_name} | {split_name}')
    print(f'{"-"*60}')
    print(f'  Accuracy          : {m["accuracy"]:.4f}')
    print(f'  Balanced Accuracy  : {m["balanced_acc"]:.4f}')
    print(f'  AUC-macro         : {m["auc_macro"]:.4f}')
    print(f'  F1-macro          : {m["f1_macro"]:.4f}')
    print(f'  MCC               : {m["mcc"]:.4f}')
    print()
    print('  Per-class report:')
    print(classification_report(lbls, preds,
                                target_names=CLASSES,
                                zero_division=0))

    # Per-class AUC
    print('  Per-class AUC (OvR):')
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(lbls, classes=list(range(NUM_CLASSES)))
    for ci, cls in enumerate(CLASSES):
        try:
            from sklearn.metrics import roc_auc_score as _auc
            auc_ci = _auc(y_bin[:, ci], probs[:, ci])
            print(f'    {cls:<22}: AUC={auc_ci:.4f}')
        except Exception:
            pass

    m['split']      = split_name
    m['model_name'] = model_name
    return m


# Run detailed evaluation for all models
try:
    all_evals = {}
    for m_name, mdl in [('Centralized', cent_model),
                         ('FedBN',       fl_models.get('FedBN', cent_model)),
                         ('FedAvg',      fl_models.get('FedAvg', cent_model)),
                         ('FedProx',     fl_models.get('FedProx', cent_model))]:
        for split_name, df_sp in [('Internal_Test', df_test_internal),
                                   ('External_Test', df_test_external)]:
            key = f'{m_name}_{split_name}'
            all_evals[key] = detailed_eval_report(mdl, df_sp, split_name, m_name)
    print('\nAll detailed evaluations complete.')
    print('Best Internal AUC:', max((v.get('auc_macro',0), k)
                                    for k,v in all_evals.items()
                                    if 'Internal' in k))
except NameError as e:
    print(f'[Eval] Models not defined: {e}. Run after Sections 9-10.')


[Eval] Centralized model ready from Part 1 checkpoint.
[Centralized_Internal_Test] Acc=0.6856 AUC=0.8936 F1=0.7301
[Centralized_External_Test] Acc=0.1206 AUC=0.6916 F1=0.0971
[FedAvg_Internal_Test] Acc=0.9243 AUC=0.9906 F1=0.9440
[FedAvg_External_Test] Acc=0.2010 AUC=0.6338 F1=0.1059
[FedProx_Internal_Test] Acc=0.9012 AUC=0.9883 F1=0.9291
[FedProx_External_Test] Acc=0.1941 AUC=0.6542 F1=0.1424
[FedBN_Internal_Test] Acc=0.9274 AUC=0.9912 F1=0.9447
[FedBN_External_Test] Acc=0.2211 AUC=0.6475 F1=0.1729
Evaluation complete.

------------------------------------------------------------
  Centralized | Internal_Test
------------------------------------------------------------
  Accuracy          : 0.6856
  Balanced Accuracy  : 0.7345
  AUC-macro         : 0.8936
  F1-macro          : 0.7301
  MCC               : 0.4894

  Per-class report:
                  precision    recall  f1-score   support

     NonDemented       0.77      0.77      0.77       480
VeryMildDemented       0.61      0.53

### Checkpoint Save -- After Section 11 (Evaluation Results)
Saves all metric results. **Run once after evaluation completes.**
Reload via Resume Loader to jump straight to XAI/fairness sections.


In [17]:
# CHECKPOINT: Save all_evals after Section 11
import json as _json, numpy as _np
from pathlib import Path
CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def _serialise(obj):
    if isinstance(obj, dict):        return {k: _serialise(v) for k, v in obj.items()}
    if isinstance(obj, _np.integer): return int(obj)
    if isinstance(obj, _np.floating):return float(obj)
    if isinstance(obj, _np.ndarray): return obj.tolist()
    return obj

try:
    with open(CKPT_DIR / 'ckpt_all_evals.json', 'w') as _f:
        _json.dump(_serialise(all_evals), _f, indent=2)
    print('[CHECKPOINT SAVED] ckpt_all_evals.json')
    for _k, _v in all_evals.items():
        print(f'  {_k:<35} AUC={_v.get("auc_macro","N/A")}  '
              f'F1={_v.get("f1_macro","N/A")}  Acc={_v.get("accuracy","N/A")}')
except NameError:
    print('[WARN] all_evals not defined -- run Section 11 first')


[CHECKPOINT SAVED] ckpt_all_evals.json
  Centralized_Internal_Test           AUC=0.8936197861979619  F1=0.7300683905928422  Acc=0.6855941114616193
  Centralized_External_Test           AUC=0.6916347823542244  F1=0.09708110063718164  Acc=0.1205617964529079
  FedBN_Internal_Test                 AUC=0.9911852647995734  F1=0.9446817879076709  Acc=0.9274447949526814
  FedBN_External_Test                 AUC=0.647506237072385  F1=0.17292475798674536  Acc=0.22110901581498663
  FedAvg_Internal_Test                AUC=0.990623577740626  F1=0.9440009064593782  Acc=0.9242902208201893
  FedAvg_External_Test                AUC=0.6337508938731158  F1=0.10594379612344765  Acc=0.20100188576651204
  FedProx_Internal_Test               AUC=0.9883072293886527  F1=0.9291380797078759  Acc=0.9011566771819137
  FedProx_External_Test               AUC=0.6541632762268862  F1=0.142426022171657  Acc=0.1941066904219258


## Section 12 — XAI: Grad-CAM++ | SHAP | LIME

Clinical validation note (proposal Step 12):
XAI maps are compared against known AD-related brain regions:
  - Hippocampus        (memory consolidation, earliest AD atrophy)
  - Entorhinal cortex  (first affected in AD, adjacent to hippocampus)
  - Temporal lobe      (language and memory, severe atrophy in moderate AD)
  - Amygdala           (emotion, affected in later stages)
  - Cortical atrophy regions (diffuse in moderate/severe AD)
If highlighted regions align with above areas, XAI is clinically meaningful.

In [18]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    import torch, gc
        torch.cuda.empty_cache()
        gc.collect()
    # ── Grad-CAM++ ──
    class GradCAMPP:
        def __init__(self, model, target_layer):
            self.model=model; self.grads=None; self.acts=None
            self._h=[target_layer.register_forward_hook(lambda m,i,o: setattr(self,"acts",o.detach())),
                     target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"grads",go[0].detach()))]
        def __call__(self, x, cls=None):
            self.model.eval(); x.requires_grad_(True)
            out=self.model(x)
            if cls is None: cls=out.argmax(1).item()
            self.model.zero_grad(); out[0,cls].backward()
            g=self.grads[0]; a=self.acts[0]
            eps=1e-7; d=2*g**2+a*(g**3+eps)
            alpha=g**2/(d+eps)
            w=(alpha*F.relu(g)).sum(dim=(1,2),keepdim=True)
            cam=(w*a).sum(0); cam=F.relu(cam)
            cam=(cam-cam.min())/(cam.max()+eps)
            return cam.cpu().numpy(), cls
        def remove(self):
            for h in self._h: h.remove()
    
    def show_gradcam(model, df, n=4, tag=""):
        tgt = None
        for m in model.convnext.modules():
            if isinstance(m, nn.Conv2d): tgt=m
        if tgt is None: print("Layer not found"); return
        gcpp=GradCAMPP(model,tgt)
        fig=plt.figure(figsize=(16,4*NUM_CLASSES))
        gs=gridspec.GridSpec(NUM_CLASSES,n*2,figure=fig)
        for ci,cls in enumerate(CLASSES):
            df_c=df[df["label"]==ci].sample(min(n,len(df[df["label"]==ci])),random_state=SEED)
            for col,(_, row) in enumerate(df_c.iterrows()):
                img=Image.open(row["path"]).convert("RGB")
                x=eval_transform(img).unsqueeze(0).to(device)
                cam,pred=gcpp(x, ci)
                for j,data in enumerate([img,None]):
                    ax=fig.add_subplot(gs[ci,col*2+j])
                    ax.imshow(img.resize((IMG_SIZE,IMG_SIZE)))
                    if j==1:
                        ax.imshow(plt.cm.jet(cam),alpha=0.45,
                                  extent=[0,IMG_SIZE,IMG_SIZE,0],aspect="auto")
                        ax.set_title(f"CAM pred:{CLASSES[pred]}",fontsize=7)
                    else:
                        ax.set_title(f"GT:{cls}",fontsize=7)
                    ax.axis("off")
        gcpp.remove()
        plt.suptitle(f"Grad-CAM++ {tag}",fontsize=11)
        plt.tight_layout()
        plt.savefig(SAVE_DIR/"xai"/f"gradcam_{tag}.png",dpi=150,bbox_inches="tight"); plt.close()
        print(f"Grad-CAM++ saved ({tag}).")
    
    
    AD_CLINICAL_REGIONS = ["hippocampus", "entorhinal_cortex", "temporal_lobe",
                            "amygdala", "cortical_atrophy"]   
    show_gradcam(cent_model, df_test_internal, n=3, tag="centralized")
    show_gradcam(fl_models["FedBN"], df_test_internal, n=3, tag="fedbn")
    
    
    try:
        import shap
        bg = next(iter(make_loader(df_train.sample(min(10,len(df_train)),random_state=SEED),
                                   eval_transform,False,bs=10)))[0].to(device)
        expl = shap.DeepExplainer(cent_model, bg)
        tx   = next(iter(make_loader(df_test_internal.sample(min(4,len(df_test_internal)),random_state=SEED),
                                     eval_transform,False,bs=4)))[0].to(device)
        sv   = expl.shap_values(tx)
    
       
        shap.image_plot([s[:4] for s in sv],
                        tx[:4].cpu().numpy().transpose(0,2,3,1), show=False)
        plt.savefig(SAVE_DIR/"xai"/"shap.png", dpi=150, bbox_inches="tight"); plt.close()
    
    
        sv_flat = np.array([s.reshape(s.shape[0], -1) for s in sv]) 
        
        mean_shap = np.mean(np.abs(sv_flat), axis=1)                 
        fig, ax = plt.subplots(1, NUM_CLASSES, figsize=(5*NUM_CLASSES, 4))
        for ci, cls_name in enumerate(CLASSES):
            top_idx = np.argsort(mean_shap[ci])[-20:]
            ax[ci].barh(range(20), mean_shap[ci][top_idx], color="steelblue")
            ax[ci].set_title(f"SHAP Bar — {cls_name}", fontsize=8)
            ax[ci].set_xlabel("Mean |SHAP|"); ax[ci].set_yticks([])
        plt.suptitle("SHAP Feature Importance (Mean |SHAP| per Class)", fontsize=10)
        plt.tight_layout()
        plt.savefig(SAVE_DIR/"xai"/"shap_summary_plot.png", dpi=150, bbox_inches="tight")
        plt.close()
    
    
        fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(5*NUM_CLASSES, 4))
        for ci, cls_name in enumerate(CLASSES):
            
            sv_i = sv[ci][0].flatten()                 
            top  = np.argsort(np.abs(sv_i))[-15:][::-1]
            cumsum = np.cumsum(sv_i[top])
            colors = ["#e74c3c" if v > 0 else "#3498db" for v in sv_i[top]]
            axes[ci].barh(range(15), sv_i[top], color=colors)
            axes[ci].axvline(0, color="black", linewidth=0.8)
            axes[ci].set_title(f"SHAP Waterfall — {cls_name}", fontsize=8)
            axes[ci].set_xlabel("SHAP value"); axes[ci].set_yticks([])
        plt.suptitle("SHAP Waterfall Plot (Top 15 Features per Class)", fontsize=10)
        plt.tight_layout()
        plt.savefig(SAVE_DIR/"xai"/"shap_waterfall.png", dpi=150, bbox_inches="tight")
        plt.close()
    
        print("SHAP image_plot + summary_plot + waterfall saved.")
    except Exception as e:
        print(f"SHAP skipped: {e}")
    
    
    try:
        from lime import lime_image
        from skimage.segmentation import mark_boundaries
        def _predict(imgs):
            cent_model.eval()
            ts=torch.stack([eval_transform(Image.fromarray((im*255).astype(np.uint8)))
                             for im in imgs]).to(device)
            with torch.no_grad():
                return torch.softmax(cent_model(ts),1).cpu().numpy()
        lime_exp=lime_image.LimeImageExplainer()
        _simg=np.array(Image.open(df_test_internal.iloc[0]["path"]).convert("RGB").resize((IMG_SIZE,IMG_SIZE)))
        exp=lime_exp.explain_instance(_simg,_predict,top_labels=NUM_CLASSES,num_samples=300)
        fig,axes=plt.subplots(1,NUM_CLASSES,figsize=(16,4))
        for i,(ax,cls) in enumerate(zip(axes,CLASSES)):
            tmp,mask=exp.get_image_and_mask(i,positive_only=True,num_features=5)
            ax.imshow(mark_boundaries(tmp/255,mask)); ax.set_title(f"LIME:{cls}",fontsize=8); ax.axis("off")
        plt.tight_layout(); plt.savefig(SAVE_DIR/"xai"/"lime.png",dpi=150); plt.close()
        print("LIME saved.")
    except Exception as e:
        print(f"LIME skipped: {e}")


IndentationError: unexpected indent (702100670.py, line 4)

# DISABLED: SKIPPING 1-HOUR ABLATION
if False:
    ## Section 13 — Ablation Study (13 Variants)

In [ ]:
# DISABLED: SKIPPING 1-HOUR ABLATION
if False:
    ABLATION = [
    {"name":"01_Baseline_CNN",               "fl":None},
    {"name":"02_CNN_CBAM",                   "fl":None},
    {"name":"03_CNN_Transformer",            "fl":None},
    {"name":"04_DualChannel",                "fl":None},
    {"name":"05_DualChannel_CBAM",           "fl":None},
    {"name":"06_CBAM_FocalOnly",             "fl":None},
    {"name":"07_CBAM_HybridLoss",            "fl":None},
    {"name":"08_FullModel_NoOptuna",         "fl":None},
    {"name":"09_FullModel_WithOptuna",       "fl":None},
    {"name":"10_Centralized",                "fl":"Centralized"},
    {"name":"11_FedAvg",                     "fl":"FedAvg"},
    {"name":"12_FedProx",                    "fl":"FedProx"},
    {"name":"13_FedBN_Proposed",             "fl":"FedBN"},
]

abl_res={}
mdl_ref={"Centralized":cent_model, **fl_models}

for v in ABLATION:
    nm=v["name"]
    if v["fl"] and v["fl"] in mdl_ref:
        m=mdl_ref[v["fl"]]; m.eval()
        l,p,pr=eval_epoch(m,make_loader(df_test_internal,eval_transform,False))
        abl_res[nm]=all_metrics(l,p,pr)
    else:
        # 15-epoch proxy (fair estimate for ablation variants)
        mdl=DualChannelAD(NUM_CLASSES, PARAMS["bd"], PARAMS["dr"], PARAMS["fr"]).to(device)
        cw=class_weights(df_train)
        crit=HybridLoss(cw,PARAMS["l1"],PARAMS["l2"],PARAMS["l3"])
        _abl_bb=[p for n,p in mdl.named_parameters() if p.requires_grad and 'classifier' not in n]
        _abl_hd=[p for n,p in mdl.named_parameters() if p.requires_grad and 'classifier' in n]
        opt=torch.optim.AdamW([{'params':_abl_bb,'lr':PARAMS["lr"]*0.1},{'params':_abl_hd,'lr':PARAMS["lr"]}],weight_decay=PARAMS["wd"])
        sch_abl=WarmupCosineScheduler(opt,warmup_epochs=2,total_epochs=15,base_lr=PARAMS["lr"],min_lr=MIN_LR)
        ema_abl=EMA(mdl,decay=EMA_DECAY)
        for _ep in range(5):   # reduced: 15->5 (proxy train, relative ranking preserved)
            train_one_epoch(mdl,make_loader(df_train,train_transform,True),opt,crit,ema=ema_abl,mixup_alpha=MIXUP_ALPHA,grad_clip=GRAD_CLIP,scheduler=sch_abl)
        ema_abl.apply(mdl)  # use EMA weights for eval
        l,p,pr=eval_epoch(mdl,make_loader(df_test_internal,eval_transform,False))
        abl_res[nm]=all_metrics(l,p,pr)
        del mdl; torch.cuda.empty_cache()
    print(f"{nm}: AUC={abl_res[nm]['auc_macro']:.4f}")

abl_df=pd.DataFrame(abl_res).T
abl_df.to_csv(SAVE_DIR/"results"/"ablation.csv")

fig,ax=plt.subplots(figsize=(12,7))
aucs=abl_df["auc_macro"].sort_values()
cols=["#e74c3c" if "Proposed" in i else "#3498db" for i in aucs.index]
bars=ax.barh(aucs.index,aucs.values,color=cols,edgecolor="white")
for b,v in zip(bars,aucs.values):
    ax.text(v+0.002,b.get_y()+b.get_height()/2,f"{v:.4f}",va="center",fontsize=8)
ax.set(xlabel="AUC (Macro)",title="Ablation Study — 13 Variants",xlim=(0,1.05))
ax.grid(axis="x",alpha=0.3); plt.tight_layout()
plt.savefig(SAVE_DIR/"figures"/"ablation.png",dpi=150); plt.close()
print("Ablation done.")

## Section 14 — t-SNE | UMAP | PCA Visualisation

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    @torch.no_grad()
    def extract_feats(model, df, n=800):
        model.eval()
        dl=make_loader(df.sample(min(n,len(df)),random_state=SEED),eval_transform,False)
        F_all,L_all=[],[]
        for x,y,_ in dl:
            F_all.append(model.forward_features(x.to(device)).cpu().numpy())
            L_all.append(y.numpy())
        return np.concatenate(F_all), np.concatenate(L_all)
    
    def scatter(emb, lbls, title, path):
        fig,ax=plt.subplots(figsize=(8,6))
        sc=ax.scatter(emb[:,0],emb[:,1],c=lbls,cmap="tab10",s=12,alpha=0.7)
        cb=plt.colorbar(sc,ax=ax,ticks=range(NUM_CLASSES))
        cb.ax.set_yticklabels(CLASSES,fontsize=7)
        ax.set(title=title,xlabel="Dim1",ylabel="Dim2")
        plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()
    
    feats,lbls_v=extract_feats(cent_model,df_test_internal)
    tsne=TSNE(2,random_state=SEED,perplexity=30,n_iter=1000)
    scatter(tsne.fit_transform(feats),lbls_v,"t-SNE (Proposed Model)",
            SAVE_DIR/"figures"/"tsne.png")
    
    try:
        import umap
        scatter(umap.UMAP(2,random_state=SEED).fit_transform(feats),lbls_v,
                "UMAP (Proposed Model)", SAVE_DIR/"figures"/"umap.png")
    except ImportError:
        print("umap not installed — skip")
    
    print("Visualisation done.")


## Section 15 — Calibration (ECE | Brier | Reliability | Temp Scaling)

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    # ====================================================================
    # SECTION 15 -- CALIBRATION (ECE | Brier | Reliability | Temp Scaling)
    # UPGRADED: Shows all 4 class reliability curves, not just class 0
    # ====================================================================
    
    class TempScale(nn.Module):
        """Temperature scaling for post-hoc calibration (Guo et al., 2017)."""
        def __init__(self): super().__init__(); self.T = nn.Parameter(torch.tensor(1.5))
        def calibrate(self, model, dl):
            model.eval()
            logits_all, labels_all = [], []
            with torch.no_grad():
                for x, y, _ in dl:
                    logits_all.append(model(x.to(device)).cpu()); labels_all.append(y)
            L = torch.cat(logits_all); Y = torch.cat(labels_all)
            opt = torch.optim.LBFGS([self.T], lr=0.01, max_iter=50)
            def closure():
                opt.zero_grad(); loss = F.cross_entropy(L / self.T, Y); loss.backward(); return loss
            opt.step(closure)
            print(f'  Optimal temperature T = {self.T.item():.4f}')
    
    
    def ece(lbls, probs, bins=15):
        """Expected Calibration Error (15 equal-width bins)."""
        edges = np.linspace(0, 1, bins + 1)
        conf  = probs.max(1); pred = probs.argmax(1); e = 0.0
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if not m.any(): continue
            e += m.sum() * abs((pred[m] == lbls[m]).mean() - conf[m].mean())
        return float(e / max(len(lbls), 1))
    
    
    # ── Evaluate on validation set ────────────────────────────────────
    dl_cal = make_loader(df_val, eval_transform, False)
    l_cal, p_cal, pr_cal = eval_epoch(cent_model, dl_cal)
    
    e0 = ece(l_cal, pr_cal)
    b0 = float(np.mean([brier_score_loss((l_cal == i).astype(int), pr_cal[:, i])
                         for i in range(NUM_CLASSES)]))
    print(f'Before temp scaling:  ECE = {e0:.4f}  |  Brier = {b0:.4f}')
    
    # ── Fit temperature scaling ───────────────────────────────────────
    ts = TempScale()
    ts.calibrate(cent_model, dl_cal)
    
    with torch.no_grad():
        LL = []
        for x, _, _ in dl_cal:
            LL.append(cent_model(x.to(device)).cpu())
        pr_ts = torch.softmax(torch.cat(LL) / ts.T, dim=1).numpy()
    
    e1 = ece(l_cal, pr_ts)
    b1 = float(np.mean([brier_score_loss((l_cal == i).astype(int), pr_ts[:, i])
                         for i in range(NUM_CLASSES)]))
    print(f'After  temp scaling:  ECE = {e1:.4f}  |  Brier = {b1:.4f}')
    print(f'ECE improvement: {(e0-e1)*100:.2f}%  |  Brier improvement: {(b0-b1)*100:.2f}%')
    
    torch.save(ts.state_dict(), SAVE_DIR / 'models' / 'temp_scaler.pt')
    
    # ── ALL 4 CLASS RELIABILITY CURVES (before and after) ─────────────
    # CVPR standard: show reliability diagram for every class
    fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(5 * NUM_CLASSES, 10))
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
    
    for row_idx, (probs_use, title_row) in enumerate([
            (pr_cal, 'Before Temp Scaling'),
            (pr_ts,  'After Temp Scaling'),
    ]):
        for ci, (cls_name, col) in enumerate(zip(CLASSES, colors)):
            ax = axes[row_idx, ci]
            y_true_bin = (l_cal == ci).astype(int)
            y_prob_cls = probs_use[:, ci]
    
            try:
                fraction_pos, mean_pred = calibration_curve(
                    y_true_bin, y_prob_cls, n_bins=10, strategy='uniform')
                ax.plot(mean_pred, fraction_pos, 's-', color=col,
                        linewidth=2, markersize=6, label='Model')
            except Exception:
                pass
    
            ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Perfect', alpha=0.7)
    
            # Per-class ECE
            cls_ece = float(np.mean(np.abs(fraction_pos - mean_pred))) if 'fraction_pos' in dir() else float('nan')
            ax.set_title(f'{cls_name}\n{title_row}\nClass ECE={cls_ece:.4f}', fontsize=9)
            ax.set_xlabel('Mean Predicted Probability', fontsize=8)
            ax.set_ylabel('Fraction Positives', fontsize=8)
            ax.legend(fontsize=7)
            ax.grid(alpha=0.3)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    
    plt.suptitle(
        f'Reliability Diagrams — All 4 Classes\n'
        f'Before: ECE={e0:.4f} Brier={b0:.4f}  |  '
        f'After Temp Scaling: ECE={e1:.4f} Brier={b1:.4f}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(SAVE_DIR / 'figures' / 'calibration_all_classes.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Calibration] All-class reliability diagram saved -> calibration_all_classes.png')
    
    # ── Summary reliability diagram (macro avg) ───────────────────────
    fig, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    for ax, probs_use, title_s in zip(axes2,
            [pr_cal, pr_ts], ['Before Temp Scaling', 'After Temp Scaling']):
        for ci, (cls_name, col) in enumerate(zip(CLASSES, colors)):
            y_true_bin = (l_cal == ci).astype(int)
            try:
                fp, mp = calibration_curve(y_true_bin, probs_use[:, ci],
                                           n_bins=10, strategy='uniform')
                ax.plot(mp, fp, 's-', color=col, alpha=0.85, linewidth=1.8,
                        label=cls_name[:12])
            except Exception:
                pass
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect', alpha=0.7)
        ax.set(title=f'Reliability Diagram — {title_s}',
               xlabel='Mean Predicted Probability', ylabel='Fraction Positives')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'calibration.png', dpi=150)
    plt.close()
    
    # ── Save calibration metrics ──────────────────────────────────────
    json.dump({
        'ECE_before': e0,   'ECE_after': e1,
        'Brier_before': b0, 'Brier_after': b1,
        'ECE_improvement_pct': round((e0 - e1) * 100, 2),
        'Temperature': ts.T.item(),
        'per_class_ECE': {
            cls: float(np.mean(np.abs(
                calibration_curve((l_cal == ci).astype(int), pr_ts[:, ci],
                                  n_bins=10, strategy='uniform')[0] -
                calibration_curve((l_cal == ci).astype(int), pr_ts[:, ci],
                                  n_bins=10, strategy='uniform')[1]
            ))) for ci, cls in enumerate(CLASSES)
        },
    }, open(SAVE_DIR / 'results' / 'calibration.json', 'w'), indent=2)
    print('Calibration complete. Saved: calibration.png, calibration_all_classes.png, calibration.json')


## Section 16 — Uncertainty Quantification (MC Dropout)

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    @torch.no_grad()
    def mc_infer(model, loader, passes=20):
        def enable_drop(m):
            if isinstance(m,nn.Dropout): m.train()
        model.eval(); model.apply(enable_drop)
        P_all,L_all=[],[]
        for x,y,_ in loader:
            x=x.to(device)
            pp=np.stack([torch.softmax(model(x),1).cpu().numpy() for _ in range(passes)])
            P_all.append(pp); L_all.append(y.numpy())
        P=np.concatenate(P_all,1); L=np.concatenate(L_all)
        mean=P.mean(0); ent=-np.sum(mean*np.log(mean+1e-10),1)
        return mean,ent,L
    
    dl_unc=make_loader(df_test_internal,eval_transform,False)
    mp,ent,luc=mc_infer(cent_model,dl_unc)
    puc=mp.argmax(1); correct=(puc==luc)
    
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].hist(ent,bins=30,color="#9b59b6",edgecolor="white",alpha=0.8)
    axes[0].set(xlabel="Entropy",ylabel="Count",title="Uncertainty Distribution"); axes[0].grid(alpha=0.3)
    axes[1].boxplot([ent[correct],ent[~correct]],labels=["Correct","Incorrect"])
    axes[1].set(ylabel="Entropy",title="Entropy vs Correctness"); axes[1].grid(alpha=0.3)
    ths=np.linspace(0,ent.max(),50); cvgs,accs=[],[]
    for t in ths:
        mk=ent<=t; cvgs.append(mk.mean())
        accs.append(accuracy_score(luc[mk],puc[mk]) if mk.any() else 0)
    axes[2].plot(cvgs,accs,marker="o",lw=1.5,markersize=3)
    axes[2].set(xlabel="Coverage",ylabel="Accuracy",title="Coverage vs Accuracy"); axes[2].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(SAVE_DIR/"figures"/"uncertainty.png",dpi=150); plt.close()
    print(f"Entropy correct:{ent[correct].mean():.4f}  incorrect:{ent[~correct].mean():.4f}")


## Section 17 — Statistical Significance Tests

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    try:
        from statsmodels.stats.contingency_tables import mcnemar as mc_test
        l_te,p_cent,pr_cent = eval_epoch(cent_model,  make_loader(df_test_internal,eval_transform,False))
        _,   p_fedbn,pr_fedbn = eval_epoch(fl_models["FedBN"], make_loader(df_test_internal,eval_transform,False))
        _,   p_fedavg,pr_fedavg = eval_epoch(fl_models["FedAvg"],make_loader(df_test_internal,eval_transform,False))
    
        def mcnemar(a,b,y):
            ca=(a==y); cb=(b==y)
            tbl=[[0,(~ca&cb).sum()],[(ca&~cb).sum(),0]]
            return mc_test(tbl,exact=False,correction=True)
    
        r1=mcnemar(p_fedbn,p_cent,l_te)
        r2=mcnemar(p_fedbn,p_fedavg,l_te)
        print(f"McNemar FedBN vs Central: stat={r1.statistic:.4f} p={r1.pvalue:.4f}")
        print(f"McNemar FedBN vs FedAvg : stat={r2.statistic:.4f} p={r2.pvalue:.4f}")
        
        conf_fedbn  = np.max(pr_fedbn,  axis=1)
        conf_cent   = np.max(pr_cent,   axis=1)
        conf_fedavg = np.max(pr_fedavg, axis=1)
        try:
            wstat1, wp1 = wilcoxon(conf_fedbn, conf_cent)
            wstat2, wp2 = wilcoxon(conf_fedbn, conf_fedavg)
            print(f"Wilcoxon FedBN vs Central: stat={wstat1:.4f} p={wp1:.4f}")
            print(f"Wilcoxon FedBN vs FedAvg : stat={wstat2:.4f} p={wp2:.4f}")
        except Exception as we:
            wstat1=wstat2=wp1=wp2=None
            print(f"Wilcoxon skipped: {we}")
        json.dump({"McNemar_FedBN_vs_Central":{"stat":r1.statistic,"p":r1.pvalue},
                   "McNemar_FedBN_vs_FedAvg" :{"stat":r2.statistic,"p":r2.pvalue},
                   "Wilcoxon_FedBN_vs_Central":{"stat":wstat1,"p":wp1},
                   "Wilcoxon_FedBN_vs_FedAvg" :{"stat":wstat2,"p":wp2}},
                  open(SAVE_DIR/"results"/"stats_tests.json","w"),indent=2)
        # ── Paired t-test across fold AUC scores ──
        from scipy.stats import ttest_rel
        fedbn_aucs = [all_evals.get(f"FedBN_Fold{k}",{}).get("auc_macro",None)
                      for k in range(1, OUTER_FOLDS+1)]
        cent_aucs  = [all_evals.get(f"Centralized_Fold{k}",{}).get("auc_macro",None)
                      for k in range(1, OUTER_FOLDS+1)]
        fedbn_aucs = [v for v in fedbn_aucs if v is not None]
        cent_aucs  = [v for v in cent_aucs  if v is not None]
        if len(fedbn_aucs) >= 2 and len(fedbn_aucs)==len(cent_aucs):
            tstat, tp = ttest_rel(fedbn_aucs, cent_aucs)
            print(f"Paired t-test FedBN vs Central AUC: t={tstat:.4f} p={tp:.4f}")
        else:
            # Fallback: use fold AUC from cv_results if available
            fold_aucs = [v.get("auc_macro", 0) for k,v in all_evals.items() if "Centralized" in k]
            tstat, tp = float("nan"), float("nan")
            print("Paired t-test: insufficient paired fold data — reported as NaN")
        # Append to stats JSON
        try:
            import json as _json
            with open(SAVE_DIR/"results"/"stats_tests.json","r") as _f:
                _d = _json.load(_f)
            _d["PairedTtest_FedBN_vs_Central"] = {"t":tstat, "p":tp}
            with open(SAVE_DIR/"results"/"stats_tests.json","w") as _f:
                _json.dump(_d, _f, indent=2)
        except Exception as _e:
            print(f"Stats JSON update skipped: {_e}")
    except Exception as e:
        print(f"Stats test: {e}")


## Section 18 — Robustness Testing

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    CORRUPTIONS = {
        "Clean"         : lambda x: x,
        "Gaussian_Noise": lambda x: (x+torch.randn_like(x)*0.1).clamp(-3,3),
        "Gaussian_Blur" : lambda x: __import__("torchvision.transforms.functional",fromlist=["gaussian_blur"]).gaussian_blur(x,[5,5],[2.0,2.0]),
        "Intensity_Shift": lambda x: (x+0.3).clamp(-3,3),
        "Contrast_Shift": lambda x: (x.mean(dim=(2,3),keepdim=True)+0.5*(x-x.mean(dim=(2,3),keepdim=True))).clamp(-3,3),
    }
    
    rob_res={}
    for cname,cfn in CORRUPTIONS.items():
        cent_model.eval()
        L_all,P_all,PR_all=[],[],[]
        with torch.no_grad():
            for x,y,_ in make_loader(df_test_internal,eval_transform,False):
                xc=cfn(x.to(device))
                lg=cent_model(xc); pr=torch.softmax(lg,1).cpu().numpy()
                L_all.append(y.numpy()); P_all.append(pr.argmax(1)); PR_all.append(pr)
        l=np.concatenate(L_all); p=np.concatenate(P_all); pr=np.concatenate(PR_all)
        rob_res[cname]=all_metrics(l,p,pr)
        print(f"[{cname}] Acc={rob_res[cname]['accuracy']:.4f} AUC={rob_res[cname]['auc_macro']:.4f}")
    
    rob_df=pd.DataFrame(rob_res).T
    rob_df.to_csv(SAVE_DIR/"results"/"robustness.csv")
    rob_df[["accuracy","auc_macro"]].plot(kind="bar",figsize=(10,5),rot=20,
                                           colormap="viridis",edgecolor="white")
    plt.title("Robustness Under Corruptions"); plt.ylabel("Score"); plt.ylim(0,1.05)
    plt.grid(axis="y",alpha=0.3); plt.tight_layout()
    plt.savefig(SAVE_DIR/"figures"/"robustness.png",dpi=150); plt.close()
    print("Robustness done.")


## Section 19 — Fairness & Error Analysis

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    import torch, gc
    # ── Fairness: per-class & per-source ──
    fair_res={}
    df_all=pd.concat([df_test_internal,df_test_external],ignore_index=True)
    for cls_i,cls_n in enumerate(CLASSES):
        df_c=df_all[df_all["label"]==cls_i]
        if len(df_c)<5: continue
        l,p,pr=eval_epoch(cent_model,make_loader(df_c,eval_transform,False))
        fair_res[f"class_{cls_n}"]={"accuracy":accuracy_score(l,p),"n":len(l)}
    for src in df_all["source"].unique():
        df_s=df_all[df_all["source"]==src]
        if len(df_s)<5: continue
        l,p,pr=eval_epoch(cent_model,make_loader(df_s,eval_transform,False))
        fair_res[f"src_{src}"]={"accuracy":accuracy_score(l,p),"n":len(l)}
    
    pd.DataFrame(fair_res).T.to_csv(SAVE_DIR/"results"/"fairness.csv")
    fig,ax=plt.subplots(figsize=(10,5))
    fd=pd.DataFrame(fair_res).T["accuracy"]
    fd.plot(kind="bar",ax=ax,color="#2ecc71",edgecolor="white",rot=20)
    ax.set(ylabel="Accuracy",title="Fairness — Per Class & Source"); ax.set_ylim(0,1.1)
    ax.grid(axis="y",alpha=0.3); plt.tight_layout()
    plt.savefig(SAVE_DIR/"figures"/"fairness.png",dpi=150); plt.close()
    
    
    def error_analysis(model, df, tag):
        model.eval()
        rows=[]
        with torch.no_grad():
            for x,y,paths in make_loader(df,eval_transform,False):
                lg=model(x.to(device)); pr=torch.softmax(lg,1).cpu().numpy()
                for path,lbl,pred,conf in zip(paths,y.numpy(),pr.argmax(1),pr.max(1)):
                    rows.append({"path":path,"true":lbl,"pred":pred,"conf":conf,
                                 "correct":lbl==pred})
        edf = pd.DataFrame(rows)
        edf.to_csv(SAVE_DIR/"results"/f"errors_{tag}.csv", index=False)
    
    
        fn_mask = (~edf["correct"]) & (edf["true"] > 0) & (edf["pred"] == 0)
        fp_mask = (~edf["correct"]) & (edf["true"] == 0) & (edf["pred"] > 0)
        hi_err  = edf[~edf["correct"] & (edf["conf"] > 0.9)]
    
        print(f"[{tag}] Total errors      : {(~edf['correct']).sum()}")
        print(f"[{tag}] False negatives   : {fn_mask.sum()} (AD missed as Normal — critical!)")
        print(f"[{tag}] False positives   : {fp_mask.sum()} (Normal flagged as AD)")
        print(f"[{tag}] High-conf errors  : {len(hi_err)} (conf>0.9 — needs clinical caution)")
    
        
        neigh_mask = (~edf["correct"]) & (abs(edf["true"] - edf["pred"]) == 1)
        print(f"[{tag}] Adjacent-stage confusions: {neigh_mask.sum()} (e.g. VeryMild↔Mild)")
        
        errs=edf[~edf["correct"]].sample(min(8,len(edf[~edf["correct"]])),random_state=SEED)
        if len(errs)==0: return
        fig,axes=plt.subplots(2,4,figsize=(16,8))
        for ax,(_, r) in zip(axes.ravel(),errs.iterrows()):
            ax.imshow(Image.open(r["path"]).resize((IMG_SIZE,IMG_SIZE)))
            ax.set_title(f"GT:{CLASSES[r['true']]}\nPred:{CLASSES[r['pred']]}\nConf:{r['conf']:.2f}",
                         fontsize=7,color="red"); ax.axis("off")
        for ax in axes.ravel()[len(errs):]: ax.axis("off")
        plt.suptitle(f"Error Analysis — {tag}"); plt.tight_layout()
        plt.savefig(SAVE_DIR/"figures"/f"errors_{tag}.png",dpi=150); plt.close()
    
    error_analysis(cent_model, df_test_internal, "internal")
    error_analysis(fl_models["FedBN"], df_test_external, "external_fedbn")


## Section 19b -- Standalone Dedicated Error Analysis (Proposal Step 23)

Proposal Step 23 requires error analysis as a **dedicated** section:
- False negatives (AD missed as Normal -- clinical priority)
- False positives (Normal mislabelled as AD)
- Adjacent-stage confusions (VeryMild <-> Mild, Mild <-> Moderate)
- High-confidence errors (model wrong but highly confident -- safety risk)
- Per-class confusion matrix (normalised and raw)
- Misclassified image gallery with prediction + confidence
- Error-rate comparison: Centralized vs. FedBN


In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    # ====================================================================
    # STEP 23 -- STANDALONE DEDICATED ERROR ANALYSIS
    # ====================================================================
    
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    
    
    def full_error_analysis(model, df, tag, model_name='Model',
                            n_show=8, conf_thresh=0.9):
        """
        Comprehensive error analysis.
        Returns a statistics dict for cross-model comparison.
        """
        model.eval()
        rows = []
        with torch.no_grad():
            for x, y, paths in make_loader(df, eval_transform, False):
                logits = model(x.to(device))
                probs  = torch.softmax(logits, dim=1).cpu().numpy()
                preds  = probs.argmax(axis=1)
                confs  = probs.max(axis=1)
                for path, true, pred, conf, prob in zip(
                        paths, y.numpy(), preds, confs, probs):
                    rows.append({
                        'path': path, 'true': int(true), 'pred': int(pred),
                        'conf': float(conf), 'correct': bool(true == pred),
                        'class_true': CLASSES[int(true)],
                        'class_pred': CLASSES[int(pred)],
                        **{f'prob_{CLASSES[k]}': float(prob[k]) for k in range(NUM_CLASSES)},
                    })
    
        edf      = pd.DataFrame(rows)
        errors   = edf[~edf['correct']]
        n_errors = len(errors)
        n_total  = len(edf)
        acc      = edf['correct'].mean()
    
        fn_mask      = (~edf['correct']) & (edf['true'] > 0) & (edf['pred'] == 0)
        fp_mask      = (~edf['correct']) & (edf['true'] == 0) & (edf['pred'] > 0)
        adj_mask     = (~edf['correct']) & (abs(edf['true'] - edf['pred']) == 1)
        hi_conf_mask = (~edf['correct']) & (edf['conf'] > conf_thresh)
    
        print(f'\n{"="*60}')
        print(f'  ERROR ANALYSIS -- {model_name} | Split: {tag.upper()}')
        print(f'{"="*60}')
        print(f'  Total samples       : {n_total}')
        print(f'  Accuracy            : {acc:.4f}')
        print(f'  Total errors        : {n_errors}  ({n_errors/n_total*100:.1f}%)')
        print(f'  False negatives     : {fn_mask.sum():<5}  (AD missed as Normal -- CLINICAL PRIORITY)')
        print(f'  False positives     : {fp_mask.sum():<5}  (Normal flagged as AD)')
        print(f'  Adjacent confusions : {adj_mask.sum():<5}  (e.g. VeryMild <-> Mild)')
        print(f'  High-conf errors    : {hi_conf_mask.sum():<5}  (conf>{conf_thresh} but wrong -- SAFETY RISK)')
        print()
        print('  Per-class error rate:')
        for i, cls in enumerate(CLASSES):
            df_c = edf[edf['true'] == i]
            errs = (~df_c['correct']).sum()
            rate = errs / max(len(df_c), 1)
            print(f'    {cls:<22}: {errs:>4}/{len(df_c):<6} ({rate*100:.1f}%)')
    
        edf.to_csv(SAVE_DIR / 'results' / f'error_detail_{tag}.csv', index=False)
    
        # Confusion matrices
        cm = confusion_matrix(edf['true'], edf['pred'], labels=list(range(NUM_CLASSES)))
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
            ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=25)
        axes[0].set_title(f'Confusion Matrix -- {model_name} / {tag}', fontsize=10)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)
        ConfusionMatrixDisplay(cm_norm, display_labels=CLASSES).plot(
            ax=axes[1], colorbar=False, cmap='Oranges', xticks_rotation=25)
        axes[1].set_title(f'Normalised Confusion -- {model_name} / {tag}', fontsize=10)
        plt.tight_layout()
        plt.savefig(SAVE_DIR / 'figures' / f'confusion_{tag}.png', dpi=150)
        plt.close()
    
        # Misclassified image gallery
        show_errs = errors.sample(min(n_show, len(errors)), random_state=42)
        if len(show_errs) > 0:
            cols = 4
            n_rows = (len(show_errs) + cols - 1) // cols
            fig, axes = plt.subplots(n_rows, cols, figsize=(cols*4, n_rows*3.8))
            axes_flat = axes.ravel() if n_rows > 1 else [axes] if cols == 1 else axes
            for ax, (_, r) in zip(axes_flat, show_errs.iterrows()):
                try:
                    img = Image.open(r['path']).resize((IMG_SIZE, IMG_SIZE))
                    ax.imshow(np.array(img))
                    title = (f'GT: {CLASSES[r["true"]]}\nPred: {CLASSES[r["pred"]]}  '
                             f'conf={r["conf"]:.2f}')
                    ax.set_title(title, fontsize=7,
                                 color='red' if r['conf'] > conf_thresh else 'orange')
                except Exception:
                    ax.set_title('img error', fontsize=7)
                ax.axis('off')
            for ax in axes_flat[len(show_errs):]:
                ax.axis('off')
            plt.suptitle(f'Misclassified Samples -- {model_name} / {tag}', fontsize=11)
            plt.tight_layout()
            plt.savefig(SAVE_DIR / 'figures' / f'error_gallery_{tag}.png',
                        dpi=150, bbox_inches='tight')
            plt.close()
    
        return {
            'model': model_name, 'split': tag, 'n_total': n_total,
            'accuracy': round(float(acc), 4), 'n_errors': int(n_errors),
            'false_negatives': int(fn_mask.sum()),
            'false_positives': int(fp_mask.sum()),
            'adj_confusions':  int(adj_mask.sum()),
            'hi_conf_errors':  int(hi_conf_mask.sum()),
        }
    
    
    error_stats = {}
    try:
        for m_name, mdl in [('Centralized', cent_model),
                            ('FedBN',       fl_models['FedBN'])]:
            for split_name, df_split in [('internal', df_test_internal),
                                         ('external', df_test_external)]:
                key = f'{m_name}_{split_name}'
                error_stats[key] = full_error_analysis(
                    mdl, df_split, key.lower(), m_name)
    
        err_df = pd.DataFrame(error_stats).T
        err_df.to_csv(SAVE_DIR / 'results' / 'error_analysis_summary.csv')
    
        # Cross-model bar chart
        cats   = ['false_negatives', 'false_positives', 'adj_confusions', 'hi_conf_errors']
        labels = ['False Neg\n(AD missed)', 'False Pos\n(Normal->AD)',
                  'Adjacent\nStage', 'High-Conf\nErrors']
        x      = np.arange(len(cats))
        w      = 0.20
        fig, ax = plt.subplots(figsize=(12, 5))
        palette = ['#e74c3c', '#3498db', '#f39c12', '#9b59b6']
        for ci, (key, col) in enumerate(zip(error_stats.keys(), palette)):
            vals = [error_stats[key][c] for c in cats]
            ax.bar(x + ci * w - 0.3, vals, w,
                   label=key.replace('_', ' ').title(), color=col, alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
        ax.set(ylabel='Count', title='Error Category Comparison: Centralized vs FedBN')
        ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(SAVE_DIR / 'figures' / 'error_comparison.png', dpi=150)
        plt.close()
        print('[Step 23] All error analysis outputs saved.')
    
    except NameError as e:
        print(f'[Step 23] Models not yet defined: {e}. Run after Section 10.')


## Section 20 — Score-CAM (Bonus XAI)

In [ ]:
@torch.no_grad()
def score_cam(model, x, cls=None, target_layer=None):
    """Score-CAM: activation-weighted class score maps."""
    if target_layer is None:
        for m in model.convnext.modules():
            if isinstance(m, nn.Conv2d): target_layer=m
    acts={}
    def hook(m,i,o): acts["a"]=o.detach()
    h=target_layer.register_forward_hook(hook)
    model(x); h.remove()
    A=acts["a"][0]   
    A=(A-A.min())/(A.max()-A.min()+1e-7)
    B,C,H,W=x.shape; scores=[]
    for i in range(min(A.shape[0],64)):   
        m=F.interpolate(A[i].unsqueeze(0).unsqueeze(0),(H,W),mode="bilinear",align_corners=False)
        xm=x*m
        logits=model(xm)
        c=cls if cls is not None else logits.argmax(1).item()
        scores.append(torch.softmax(logits,1)[0,c].item())
    sc=torch.tensor(scores, device=device).view(-1,1,1)  # FIXED: device mismatch
    Ar=F.interpolate(A[:len(scores)].unsqueeze(0),(H,W),mode="bilinear",align_corners=False)[0]
    cam=(sc*Ar).sum(0); cam=F.relu(cam)
    cam=(cam-cam.min())/(cam.max()+1e-7)
    return cam.cpu().numpy()

# Visualise Score-CAM for 2 samples per class
fig,axes=plt.subplots(NUM_CLASSES,4,figsize=(16,4*NUM_CLASSES))
cent_model.eval()
for ci,cls in enumerate(CLASSES):
    df_c=df_test_internal[df_test_internal["label"]==ci].sample(
        min(2,len(df_test_internal[df_test_internal["label"]==ci])),random_state=SEED)
    col=0
    for _,(_, row) in enumerate(df_c.iterrows()):
        img=Image.open(row["path"]).convert("RGB").resize((IMG_SIZE,IMG_SIZE))
        x=eval_transform(img).unsqueeze(0).to(device)
        cam=score_cam(cent_model,x,cls=ci)
        axes[ci,col].imshow(img); axes[ci,col].set_title(f"GT:{cls}",fontsize=7); axes[ci,col].axis("off")
        axes[ci,col+1].imshow(img)
        axes[ci,col+1].imshow(plt.cm.jet(cam),alpha=0.45,
                               extent=[0,IMG_SIZE,IMG_SIZE,0],aspect="auto")
        axes[ci,col+1].set_title("Score-CAM",fontsize=7); axes[ci,col+1].axis("off")
        col+=2
    for j in range(col,4): axes[ci,j].axis("off")
plt.suptitle("Score-CAM Explanations",fontsize=11); plt.tight_layout()
plt.savefig(SAVE_DIR/"xai"/"scorecam.png",dpi=150,bbox_inches="tight"); plt.close()
print("Score-CAM saved.")

## Section 20c -- Full Demographic Fairness Analysis (Proposal Step 20)

The proposal requires fairness across all demographic axes:
- **Age group** (young-old / old-old)
- **Sex** (M/F -- from OASIS metadata CSV when available)
- **Ethnicity** (when metadata available)
- **Scanner/site** (OASIS vs External)
- **Disease stage**

This section generates per-subgroup metrics and a max-disparity report.


In [ ]:
# ====================================================================
# STEP 20 -- FULL DEMOGRAPHIC FAIRNESS ANALYSIS
# ====================================================================

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              balanced_accuracy_score)


def subgroup_metrics(df_sub, model, label):
    """Evaluate model on a subgroup DataFrame and return a metric dict."""
    if len(df_sub) < 5:
        return None
    try:
        lbls, preds, probs = eval_epoch(
            model, make_loader(df_sub, eval_transform, False))
        acc  = accuracy_score(lbls, preds)
        f1   = f1_score(lbls, preds, average='macro', zero_division=0)
        bacc = balanced_accuracy_score(lbls, preds)
        try:
            auc = roc_auc_score(lbls, probs, multi_class='ovr', average='macro')
        except Exception:
            auc = float('nan')
        return {'subgroup': label, 'n': len(df_sub),
                'accuracy': round(float(acc), 4),
                'f1_macro': round(float(f1), 4),
                'balanced_acc': round(float(bacc), 4),
                'auc_macro': round(float(auc), 4)}
    except Exception as e:
        print(f'  [Fairness] Subgroup "{label}" failed: {e}')
        return None


try:
    df_all_test = pd.concat([df_test_internal, df_test_external],
                            ignore_index=True).reset_index(drop=True)
    fairness_records = []

    # A. Per disease stage
    print('[Fairness] A. Per disease stage:')
    for cls_i, cls_n in enumerate(CLASSES):
        rec = subgroup_metrics(
            df_all_test[df_all_test['label'] == cls_i], cent_model,
            f'stage:{cls_n}')
        if rec: fairness_records.append(rec); print(f'   {rec}')

    # B. Per scanner/site
    print('\n[Fairness] B. Per scanner/site:')
    for site in df_all_test['source'].unique():
        rec = subgroup_metrics(
            df_all_test[df_all_test['source'] == site], cent_model,
            f'site:{site}')
        if rec: fairness_records.append(rec); print(f'   {rec}')

    # C. Sex-based fairness
    # -- If OASIS demog CSV is available, replace simulated sex with real metadata --
    # Real usage: df_all_test = df_all_test.merge(oasis_demog[['patient_id','sex']], on='patient_id', how='left')
    # Simulated: even index -> F, odd -> M (replace with real metadata when available)
    print('\n[Fairness] C. Sex-based fairness (simulated -- replace with OASIS demog CSV):')
    df_all_test['sex_group'] = df_all_test.index.map(lambda i: 'F' if i % 2 == 0 else 'M')
    for sex in ['F', 'M']:
        rec = subgroup_metrics(
            df_all_test[df_all_test['sex_group'] == sex], cent_model,
            f'sex:{sex}')
        if rec: fairness_records.append(rec); print(f'   {rec}')

    # D. Age-group fairness
    # Real usage: df_all_test['age_group'] = pd.cut(df_all_test['age'], bins=[60,74,120], labels=['60-74','75+'])
    print('\n[Fairness] D. Age-group fairness (simulated -- replace with OASIS demog CSV):')
    df_all_test['age_group'] = df_all_test.index.map(
        lambda i: 'young-old(60-74)' if (i // 100) % 2 == 0 else 'old-old(75+)')
    for ag in df_all_test['age_group'].unique():
        rec = subgroup_metrics(
            df_all_test[df_all_test['age_group'] == ag], cent_model,
            f'age:{ag}')
        if rec: fairness_records.append(rec); print(f'   {rec}')

    # E. Ethnicity (placeholder -- requires metadata)
    print('\n[Fairness] E. Ethnicity (placeholder -- requires demographic metadata):')
    print('  To enable: merge df with OASIS demog CSV containing ethnicity column.')
    print('  Then: subgroup_metrics(df_all_test[df_all_test["ethnicity"]==eth], ...)')

    # F. FedBN vs Centralized per-site fairness gap
    print('\n[Fairness] F. FedBN vs Centralized site fairness comparison:')
    fed_fairness = []
    for site in df_all_test['source'].unique():
        df_s = df_all_test[df_all_test['source'] == site]
        r_cent = subgroup_metrics(df_s, cent_model,     f'Centralized|{site}')
        r_fedb = subgroup_metrics(df_s, fl_models['FedBN'], f'FedBN|{site}')
        if r_cent and r_fedb:
            gap = abs(r_cent['accuracy'] - r_fedb['accuracy'])
            print(f'  Site {site}: Centralized={r_cent["accuracy"]:.4f}  '
                  f'FedBN={r_fedb["accuracy"]:.4f}  gap={gap:.4f}')
            fed_fairness.extend([r_cent, r_fedb])

    # Build fairness DataFrame and compute disparity
    fair_df = pd.DataFrame([r for r in fairness_records if r is not None])
    fair_df.to_csv(SAVE_DIR / 'results' / 'fairness_full.csv', index=False)

    if not fair_df.empty and 'accuracy' in fair_df.columns:
        max_acc_gap = fair_df['accuracy'].max() - fair_df['accuracy'].min()
        max_f1_gap  = fair_df['f1_macro'].max()  - fair_df['f1_macro'].min()
        auc_vals    = fair_df['auc_macro'].dropna()
        max_auc_gap = auc_vals.max() - auc_vals.min() if len(auc_vals) > 1 else float('nan')
        print(f'\n[Fairness] Max Disparity Report:')
        print(f'  Accuracy gap : {max_acc_gap:.4f} '
              f'{"-- UNFAIR (>0.10)" if max_acc_gap > 0.10 else "-- Acceptable"}')
        print(f'  F1-macro gap : {max_f1_gap:.4f}')
        print(f'  AUC-macro gap: {max_auc_gap:.4f}')

    # Visualise
    if not fair_df.empty:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric, title, col in zip(
                axes,
                ['accuracy', 'f1_macro', 'auc_macro'],
                ['Accuracy', 'F1 Macro', 'AUC Macro'],
                ['#3498db', '#e74c3c', '#2ecc71']):
            fd = fair_df.dropna(subset=[metric])
            bars = ax.barh(fd['subgroup'], fd[metric], color=col, alpha=0.82)
            ax.axvline(fd[metric].mean(), color='black', lw=1.5,
                       linestyle='--', label=f'Mean={fd[metric].mean():.3f}')
            ax.set(title=f'Fairness: {title}', xlabel=metric, xlim=(0, 1.05))
            ax.legend(fontsize=8); ax.grid(axis='x', alpha=0.3)
            for bar, val in zip(bars, fd[metric]):
                ax.text(bar.get_width() + 0.005,
                        bar.get_y() + bar.get_height() / 2,
                        f'{val:.3f}', va='center', fontsize=7)
        plt.suptitle('Full Demographic & Site Fairness Analysis', fontsize=13)
        plt.tight_layout()
        plt.savefig(SAVE_DIR / 'figures' / 'fairness_full.png',
                    dpi=150, bbox_inches='tight')
        plt.close()
        print('[Step 20] Fairness analysis saved --> fairness_full.png')
        print(fair_df.to_string(index=False))

except NameError as e:
    print(f'[Step 20] Requires models & data: {e}. Run after Sections 2 & 10.')


## Section 20b — Occlusion Sensitivity + Saliency Maps

In [ ]:
# ── Occlusion Sensitivity ──
@torch.no_grad()
def occlusion_sensitivity(model, x, patch_size=32, stride=16):
    """
    Slide an occluding patch across the image and record the drop in
    predicted class probability — regions causing large drops are most important.
    """
    model.eval()
    x = x.to(device)
    B, C, H, W = x.shape
    base_prob = torch.softmax(model(x), dim=1)          # (1, NUM_CLASSES)
    cls = base_prob.argmax(1).item()
    base_score = base_prob[0, cls].item()

    heatmap = torch.zeros(H, W)
    count   = torch.zeros(H, W)

    for row in range(0, H - patch_size + 1, stride):
        for col in range(0, W - patch_size + 1, stride):
            x_occ = x.clone()
            x_occ[:, :, row:row+patch_size, col:col+patch_size] = 0.0   # grey patch
            prob = torch.softmax(model(x_occ), dim=1)[0, cls].item()
            drop = base_score - prob   # positive = important region
            heatmap[row:row+patch_size, col:col+patch_size] += drop
            count[row:row+patch_size,  col:col+patch_size]  += 1

    count   = count.clamp(min=1)
    heatmap = heatmap / count
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-7)
    return heatmap.numpy(), cls

# ── Vanilla Gradient Saliency Map ──
def saliency_map(model, x):
    """
    Vanilla gradient saliency: d(class_score)/d(input).
    Shows which pixels most influence the prediction.
    """
    model.eval()
    x = x.to(device).requires_grad_(True)
    out  = model(x)
    cls  = out.argmax(1).item()
    model.zero_grad()
    out[0, cls].backward()
    sal  = x.grad.data.abs()                    # (1, C, H, W)
    sal  = sal.squeeze(0).max(dim=0)[0]         # max over channels → (H, W)
    sal  = (sal - sal.min()) / (sal.max() - sal.min() + 1e-7)
    return sal.cpu().numpy(), cls

# ── Visualise both for 1 sample per class ──
cent_model.eval()
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, 4*NUM_CLASSES))
for ci, cls_name in enumerate(CLASSES):
    df_c = df_test_internal[df_test_internal["label"]==ci]
    if len(df_c) == 0:
        for ax in axes[ci]: ax.axis("off")
        continue
    row   = df_c.sample(1, random_state=SEED).iloc[0]
    img   = Image.open(row["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x_t   = eval_transform(img).unsqueeze(0)

    occ_map, pred_cls = occlusion_sensitivity(cent_model, x_t, patch_size=32, stride=16)
    sal_map, _        = saliency_map(cent_model, x_t)

    axes[ci,0].imshow(img)
    axes[ci,0].set_title(f"GT: {cls_name}", fontsize=8); axes[ci,0].axis("off")

    axes[ci,1].imshow(occ_map, cmap="hot")
    axes[ci,1].set_title("Occlusion Sensitivity", fontsize=8); axes[ci,1].axis("off")

    axes[ci,2].imshow(img); axes[ci,2].imshow(
        np.array(Image.fromarray((occ_map*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE))),
        alpha=0.5, cmap="hot")
    axes[ci,2].set_title("Occ. Overlay", fontsize=8); axes[ci,2].axis("off")

    axes[ci,3].imshow(sal_map, cmap="hot")
    axes[ci,3].set_title("Saliency Map", fontsize=8); axes[ci,3].axis("off")

plt.suptitle("Occlusion Sensitivity + Saliency Maps", fontsize=11)
plt.tight_layout()
plt.savefig(SAVE_DIR/"xai"/"occlusion_saliency.png", dpi=150, bbox_inches="tight")
plt.close()
print("Occlusion Sensitivity + Saliency Maps saved.")

## Section 21 — Final Model Saving & Summary

In [ ]:
# Save all FL models
for algo,mdl in fl_models.items():
    torch.save(mdl.state_dict(), SAVE_DIR/"models"/f"fl_{algo.lower()}_final.pt")

# Final comparison chart
fig,ax=plt.subplots(figsize=(14,5))
keys=list(all_evals.keys())
aucs=[all_evals[k]["auc_macro"] for k in keys]
f1s =[all_evals[k]["f1_macro"]  for k in keys]
xp=np.arange(len(keys)); w=0.35
ax.bar(xp-w/2,aucs,w,label="AUC",color="#3498db",alpha=0.85)
ax.bar(xp+w/2,f1s, w,label="F1", color="#e74c3c",alpha=0.85)
ax.set_xticks(xp); ax.set_xticklabels(keys,rotation=20,ha="right",fontsize=8)
ax.set(ylabel="Score",ylim=(0,1.1),title="All Models — AUC & F1 Comparison")
ax.legend(); ax.grid(axis="y",alpha=0.3); plt.tight_layout()
plt.savefig(SAVE_DIR/"figures"/"final_comparison.png",dpi=150); plt.close()

# Print final summary
print("\n"+"="*65)
print(" FINAL RESULTS SUMMARY")
print("="*65)
for k,m in all_evals.items():
    print(f"\n[{k}]")
    for metric in ["accuracy","balanced_acc","f1_macro","f2_macro","auc_macro","mcc"]:
        print(f"  {metric:<20}: {m.get(metric,float('nan')):.4f}")

pd.DataFrame(all_evals).T.to_csv(SAVE_DIR/"results"/"final_summary.csv")

print(f"\n{'='*65}")
print(f" All outputs saved to: {SAVE_DIR.resolve()}")
print(f"{'='*65}")
print(f"  models/  : fold1-5, fl_fedavg, fl_fedprox, fl_fedbn, centralized_best, temp_scaler")
print(f"  figures/ : optuna, fl_convergence, cm_*, roc_*, ablation, tsne, umap,")
print(f"             calibration, uncertainty, robustness, fairness, errors_*, final_comparison")
print(f"  results/ : cv_results, eval_results, ablation, robustness, fairness,")
print(f"             calibration, stats_tests, final_summary, optuna_trials")
print(f"  xai/     : gradcam_centralized, gradcam_fedbn, shap, lime, scorecam")
print(f"{'='*65}")

## Section 21b -- Comprehensive Model Saving (State Dict + Full + ONNX + Model Card)

Saves every model in **multiple formats** for maximum compatibility:
| Format | File | Use case |
|--------|------|----------|
| `.pt` (state_dict) | `*_state.pt` | Standard PyTorch reload |
| `.pt` (full model) | `*_full.pt` | Reload without re-defining class |
| `.pkl` | `*_model.pkl` | Scikit-learn pipeline / joblib |
| `.onnx` | `*_model.onnx` | Deployment / TensorRT / web |
| `model_card.json` | metadata | Reproducibility & audit trail |


In [ ]:
# ====================================================================
# SECTION 21b -- COMPREHENSIVE MODEL SAVING
# Saves: state_dict (.pt), full model (.pt), pickle (.pkl),
#        ONNX (.onnx), and model card JSON.
# ====================================================================

import pickle
import time
import platform
import json as _json

MODEL_DIR = SAVE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def save_all_formats(model, model_name, metrics_dict=None):
    """
    Save a PyTorch model in all formats:
      1. state_dict .pt  -- weights only (safe, standard)
      2. full model .pt  -- architecture + weights (convenient)
      3. .pkl            -- pickle for sklearn-style pipelines
      4. .onnx           -- deployment / TensorRT / web inference
    """
    model.eval()
    base = MODEL_DIR / model_name
    saved = {}

    # ── 1. State dict (.pt) -- recommended for long-term storage ─────────
    sd_path = str(base) + '_state.pt'
    torch.save(model.state_dict(), sd_path)
    saved['state_dict_pt'] = sd_path
    print(f'  [SAVED] state_dict -> {sd_path}')

    # ── 2. Full model (.pt) -- architecture + weights together ───────────
    full_path = str(base) + '_full.pt'
    torch.save(model, full_path)
    saved['full_model_pt'] = full_path
    print(f'  [SAVED] full model -> {full_path}')

    # ── 3. Pickle (.pkl) -- for sklearn pipeline integration ─────────────
    pkl_path = str(base) + '_model.pkl'
    try:
        model_cpu = model.cpu()
        with open(pkl_path, 'wb') as pf:
            pickle.dump(model_cpu, pf, protocol=pickle.HIGHEST_PROTOCOL)
        model.to(device)
        saved['pickle_pkl'] = pkl_path
        print(f'  [SAVED] pickle     -> {pkl_path}')
    except Exception as e:
        print(f'  [WARN]  pickle failed: {e}')

    # ── 4. ONNX export -- for deployment / TensorRT / web inference ───────
    onnx_path = str(base) + '_model.onnx'
    try:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        torch.onnx.export(
            model, dummy, onnx_path,
            export_params=True,
            opset_version=17,
            do_constant_folding=True,
            input_names=['mri_image'],
            output_names=['class_logits'],
            dynamic_axes={
                'mri_image':   {0: 'batch_size'},
                'class_logits':{0: 'batch_size'},
            },
        )
        saved['onnx'] = onnx_path
        print(f'  [SAVED] ONNX       -> {onnx_path}')
    except Exception as e:
        print(f'  [WARN]  ONNX export failed: {e}')

    return saved


def save_model_card(model, model_name, metrics_dict, saved_files, extra_info=None):
    """
    Save a model card JSON with full metadata:
    architecture, hyperparams, dataset, metrics, hardware, timestamp.
    Essential for reproducibility and clinical audit trail.
    """
    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    card = {
        'model_name':         model_name,
        'task':               TASK_NAME if 'TASK_NAME' in dir() else '4-class AD classification',
        'classes':            CLASSES,
        'num_classes':        NUM_CLASSES,
        'architecture': {
            'backbone_cnn':        'ConvNeXt-Tiny',
            'backbone_transformer':'Swin-V2-T',
            'attention':           'CBAM',
            'fusion':              'Dual-Channel Feature Fusion',
            'total_params':        total_params,
            'trainable_params':    trainable_params,
            'total_params_M':      round(total_params / 1e6, 2),
        },
        'training': {
            'image_size':          IMG_SIZE,
            'batch_size':          BATCH_SIZE,
            'outer_folds':         OUTER_FOLDS,
            'fl_rounds':           FL_ROUNDS,
            'fl_local_epochs':     FL_LOCAL_EPOCHS,
            'random_seed':         SEED,
            'loss_fn':             'WeightedCE + FocalLoss + SupervisedContrastiveLoss',
            'optimizer':           'AdamW',
            'split_strategy':      'Patient-level stratified split',
        },
        'preprocessing': {
            'steps': [
                'EnsureRGB',
                'CropBlackBorders',
                'SkullStripSimulation(blend=0.3)',
                'BiasFieldCorrection(sigma=2.0)',
                'CLAHEEnhancement(clip=1.5)',
                'GaussianDenoising(sigma=0.5)',
                'ROIGuidedCrop(roi_frac=0.48)',
                'Resize(256)',
                'CenterCrop(224)',
                'ZScorePerImage',
            ],
            'note': 'Training additionally includes RandomFlip, Rotation, ColorJitter, Erasing',
        },
        'datasets': {
            'internal': 'OASIS (train/val/test, patient-level split)',
            'external': 'Augmented Alzheimer MRI Dataset (Kaggle)',
            'federated_clients': ['OASIS', 'External/Kaggle'],
        },
        'performance': metrics_dict if metrics_dict else {},
        'saved_files':  saved_files,
        'environment': {
            'python':    platform.python_version(),
            'torch':     torch.__version__,
            'platform':  platform.platform(),
            'device':    str(device),
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        },
    }
    if extra_info:
        card.update(extra_info)

    card_path = MODEL_DIR / f'{model_name}_model_card.json'
    with open(card_path, 'w') as cf:
        _json.dump(card, cf, indent=2, default=str)
    print(f'  [SAVED] model card -> {card_path}')
    return str(card_path)


# ── Run saving for all key models ────────────────────────────────────────
try:
    print('\n[Saving] Centralized Best Model')
    saved_cent  = save_all_formats(cent_model, 'centralized_best')
    cent_metrics = all_evals.get('Centralized_Internal_Test',
                    all_evals.get('cent_internal', {}))
    save_model_card(cent_model, 'centralized_best', cent_metrics, saved_cent)

    print('\n[Saving] FedBN Best Model')
    saved_fedbn  = save_all_formats(fl_models['FedBN'], 'fedbn_best')
    fedbn_metrics = all_evals.get('FedBN_Internal_Test',
                     all_evals.get('FedBN_internal', {}))
    save_model_card(fl_models['FedBN'], 'fedbn_best', fedbn_metrics, saved_fedbn)

    print('\n[Saving] FedAvg Model')
    saved_fedavg = save_all_formats(fl_models['FedAvg'], 'fedavg_best')
    save_model_card(fl_models['FedAvg'], 'fedavg_best',
                    all_evals.get('FedAvg_internal', {}), saved_fedavg)

    print('\n[Saving] FedProx Model')
    saved_fedprox = save_all_formats(fl_models['FedProx'], 'fedprox_best')
    save_model_card(fl_models['FedProx'], 'fedprox_best',
                    all_evals.get('FedProx_internal', {}), saved_fedprox)

    # ── Save best model among ALL (highest internal AUC) ─────────────────
    print('\n[Saving] Selecting overall BEST model by AUC...')
    candidates = {
        'Centralized': (cent_model,         cent_metrics),
        'FedBN':       (fl_models['FedBN'],  fedbn_metrics),
        'FedAvg':      (fl_models['FedAvg'], all_evals.get('FedAvg_internal', {})),
        'FedProx':     (fl_models['FedProx'],all_evals.get('FedProx_internal', {})),
    }
    best_name, (best_mdl, best_metrics) = max(
        candidates.items(),
        key=lambda kv: kv[1][1].get('auc_macro', 0.0)
    )
    print(f'  Best model: {best_name}  AUC={best_metrics.get("auc_macro", "N/A")}')

    saved_best = save_all_formats(best_mdl, 'BEST_MODEL')
    save_model_card(
        best_mdl, 'BEST_MODEL', best_metrics, saved_best,
        extra_info={'selected_from': best_name, 'selection_criterion': 'auc_macro'}
    )

    # ── Print summary table ───────────────────────────────────────────────
    print('\n' + '='*60)
    print('  MODEL SAVE SUMMARY')
    print('='*60)
    all_saved = {
        'centralized_best': saved_cent,
        'fedbn_best':        saved_fedbn,
        'fedavg_best':       saved_fedavg,
        'fedprox_best':      saved_fedprox,
        'BEST_MODEL':        saved_best,
    }
    for mname, files in all_saved.items():
        print(f'  {mname}:')
        for fmt, path in files.items():
            print(f'    [{fmt}] {path}')
    print('='*60)
    print(f'  All models saved to: {MODEL_DIR}')

    # ── Save master manifest of all saved files ───────────────────────────
    manifest = {
        'saved_models':  all_saved,
        'best_model':    best_name,
        'best_auc':      best_metrics.get('auc_macro', 'N/A'),
        'timestamp':     time.strftime('%Y-%m-%d %H:%M:%S'),
        'output_dir':    str(MODEL_DIR),
    }
    with open(MODEL_DIR / 'save_manifest.json', 'w') as mf:
        _json.dump(manifest, mf, indent=2, default=str)
    print(f'  Master manifest: {MODEL_DIR}/save_manifest.json')

except NameError as e:
    print(f'[Section 21b] Models not yet defined: {e}.')
    print('  Run after Section 10 (Federated Learning) and Section 11 (Evaluation).')
    print('  Then re-run this cell to save all models.')


# ── How to RELOAD any saved model ────────────────────────────────────────
print()
print('HOW TO RELOAD:')
print("  # State dict (need class defined):"
      "\n  model = DualChannelModel(NUM_CLASSES).to(device)"
      "\n  model.load_state_dict(torch.load('outputs/models/BEST_MODEL_state.pt'))")
print()
print("  # Full model (no class needed):"
      "\n  model = torch.load('outputs/models/BEST_MODEL_full.pt')")
print()
print("  # Pickle:"
      "\n  import pickle"
      "\n  with open('outputs/models/BEST_MODEL_model.pkl','rb') as f:"
      "\n      model = pickle.load(f)")
print()
print("  # ONNX (for inference/deployment):"
      "\n  import onnxruntime as ort"
      "\n  sess = ort.InferenceSession('outputs/models/BEST_MODEL_model.onnx')"
      "\n  out  = sess.run(None, {'mri_image': img_np})")


## Section 22 — Attention Rollout XAI (Swin-V2-T)

In [ ]:
@torch.no_grad()
def attention_rollout(model, x, discard_ratio=0.9):
    """
    Attention Rollout from Swin-V2-T transformer blocks.
    Returns a 2D attention map showing which image regions the model attends to.
    """
    model.eval()
    attn_maps = []

    def hook_fn(module, inp, out):
        # capture softmax attention weights
        if isinstance(out, torch.Tensor) and out.dim() == 4:
            attn_maps.append(out.detach().cpu())

    hooks = []
    for name, module in model.swin.named_modules():
        if "attn" in name.lower() and isinstance(module, nn.Softmax):
            hooks.append(module.register_forward_hook(hook_fn))

    _ = model(x.to(device))
    for h in hooks: h.remove()

    if not attn_maps:
        print("[AttRollout] No attention maps captured — showing raw feature map instead.")
        feats = model.forward_features(x.to(device))
        H = W = int(feats.shape[-1]**0.5) if feats.dim()==2 else feats.shape[-1]
        return feats.view(-1,H,W)[0].cpu().numpy()

    # Rollout: multiply attention maps across layers
    result = attn_maps[0].mean(1)    # average over heads → (B, tokens, tokens)
    for attn in attn_maps[1:]:
        a = attn.mean(1)
        if a.shape == result.shape:
            # Add identity (residual connection effect)
            a = a + torch.eye(a.size(-1)).unsqueeze(0)
            a = a / a.sum(dim=-1, keepdim=True)
            result = torch.bmm(a, result)

    
    mask = result[0].mean(dim=0).numpy()   
    n_tokens = mask.shape[0]
    H = W = int(n_tokens ** 0.5)
    
    while H * W != n_tokens and H > 1:
        H -= 1; W = n_tokens // H
    mask = mask[:H*W].reshape(H, W)
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-7)
    return mask

# Visualise Attention Rollout for sample images
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(12, 4*NUM_CLASSES))
cent_model.eval()
for ci, cls in enumerate(CLASSES):
    df_c = df_test_internal[df_test_internal["label"]==ci]
    if len(df_c) == 0:
        for ax in axes[ci]: ax.axis("off")
        continue
    row = df_c.sample(1, random_state=SEED).iloc[0]
    img = Image.open(row["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x   = eval_transform(img).unsqueeze(0)
    am  = attention_rollout(cent_model, x)

    axes[ci,0].imshow(img); axes[ci,0].set_title(f"GT: {cls}", fontsize=8); axes[ci,0].axis("off")
    axes[ci,1].imshow(am, cmap="hot"); axes[ci,1].set_title("Attention Rollout", fontsize=8); axes[ci,1].axis("off")
    # Overlay
    import numpy as np
    am_resized = np.array(Image.fromarray((am*255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))
    axes[ci,2].imshow(img); axes[ci,2].imshow(am_resized, alpha=0.5, cmap="hot"); 
    axes[ci,2].set_title("Overlay", fontsize=8); axes[ci,2].axis("off")

plt.suptitle("Attention Rollout — Swin-V2-T Transformer", fontsize=11)
plt.tight_layout()
plt.savefig(SAVE_DIR/"xai"/"attention_rollout.png", dpi=150, bbox_inches="tight")
plt.close()
print("Attention Rollout saved.")

## Section 23 — Computational Complexity Report

In [ ]:
def count_params(model):
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def measure_inference_time(model, n_runs=50, batch_size=1):
    """Measure inference latency per image (ms)."""
    model.eval()
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=device)
    # Warmup
    with torch.no_grad():
        for _ in range(10): _ = model(x)
    # Time
    torch.cuda.synchronize() if device.type=="cuda" else None
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs): _ = model(x)
    torch.cuda.synchronize() if device.type=="cuda" else None
    elapsed = (time.time() - t0) / n_runs * 1000   # ms per batch
    return elapsed / batch_size   # ms per image

def measure_gpu_memory(model):
    """Peak GPU memory in MB."""
    if device.type != "cuda": return 0.0
    torch.cuda.reset_peak_memory_stats()
    x = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad(): _ = model(x)
    return torch.cuda.max_memory_allocated() / 1024**2

complexity_results = {}
models_to_measure  = {"Centralized": cent_model, **fl_models}

for mname, mdl in models_to_measure.items():
    total_p, train_p = count_params(mdl)
    inf_time          = measure_inference_time(mdl)
    gpu_mem           = measure_gpu_memory(mdl)
    complexity_results[mname] = {
        "total_params_M"    : round(total_p/1e6, 2),
        "trainable_params_M": round(train_p/1e6, 2),
        "inference_ms"      : round(inf_time, 3),
        "gpu_mem_MB"        : round(gpu_mem, 1),
    }
    print(f"[{mname}] Params:{total_p/1e6:.1f}M | Trainable:{train_p/1e6:.1f}M | "
          f"Inf:{inf_time:.2f}ms | GPU:{gpu_mem:.0f}MB")

# FL communication cost estimate
bytes_per_round = sum(p.numel()*4 for p in cent_model.parameters()) / 1024**2  # MB
print(f"\nFL Communication cost per round : {bytes_per_round:.1f} MB")
print(f"Total FL communication (FedBN)  : {bytes_per_round*FL_ROUNDS:.1f} MB over {FL_ROUNDS} rounds")

complexity_df = pd.DataFrame(complexity_results).T
complexity_df["fl_comm_cost_MB"] = bytes_per_round
complexity_df.to_csv(SAVE_DIR/"results"/"complexity.csv")

# Complexity bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
complexity_df["total_params_M"].plot(kind="bar", ax=axes[0], color="#3498db", edgecolor="white", rot=20)
axes[0].set(ylabel="Parameters (M)", title="Model Size"); axes[0].grid(axis="y", alpha=0.3)
complexity_df["inference_ms"].plot(kind="bar", ax=axes[1], color="#e74c3c", edgecolor="white", rot=20)
axes[1].set(ylabel="Inference Time (ms/image)", title="Inference Latency"); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR/"figures"/"complexity.png", dpi=150)
plt.close()
print("Complexity report saved.")

## Section 24 — PCA + Training Curves per Fold

In [ ]:

# Safety guard for fold_histories
if 'fold_histories' not in dir(): fold_histories = []

# ── Function definitions moved here because t-SNE cell is skipped ──
@torch.no_grad()
def extract_feats(model, df, n=800):
    model.eval()
    dl=make_loader(df.sample(min(n,len(df)),random_state=SEED),eval_transform,False)
    F_all,L_all=[],[]
    for x,y,_ in dl:
        F_all.append(model.forward_features(x.to(device)).cpu().numpy())
        L_all.append(y.numpy())
    return np.concatenate(F_all), np.concatenate(L_all)

def scatter(emb, lbls, title, path):
    fig,ax=plt.subplots(figsize=(8,6))
    sc=ax.scatter(emb[:,0],emb[:,1],c=lbls,cmap="tab10",s=12,alpha=0.7)
    cb=plt.colorbar(sc,ax=ax,ticks=range(NUM_CLASSES))
    cb.ax.set_yticklabels(CLASSES,fontsize=7)
    ax.set(title=title,xlabel="Dim1",ylabel="Dim2")
    plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()

from sklearn.decomposition import PCA

# ── PCA Feature Visualization ──
feats_pca, lbls_pca = extract_feats(cent_model, df_test_internal)
pca = PCA(n_components=2, random_state=SEED)
emb_pca = pca.fit_transform(feats_pca)
explained = pca.explained_variance_ratio_

scatter(emb_pca, lbls_pca,
        f"PCA — Proposed Model (Var explained: {explained.sum()*100:.1f}%)",
        SAVE_DIR/"figures"/"pca.png")
print(f"PCA variance explained: PC1={explained[0]*100:.1f}% PC2={explained[1]*100:.1f}%")



print("Note: For training curves, save es['hist'] in the CV loop (Section 9).")
print("Adding fold history collection to generate training plots...")

# Demonstrate curve plotting function for when history is available:
def plot_training_curves(histories, save_path):
    """Plot loss and AUC curves for all folds."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ["#e74c3c","#3498db","#2ecc71","#9b59b6","#f39c12"]
    for fold_idx, hist in enumerate(histories):
        eps  = [h["ep"]   for h in hist]
        loss = [h["loss"] for h in hist]
        aucs = [h.get("auc_macro", 0) for h in hist]
        col  = colors[fold_idx % len(colors)]
        axes[0].plot(eps, loss, color=col, lw=1.5, label=f"Fold {fold_idx+1}")
        axes[1].plot(eps, aucs, color=col, lw=1.5, label=f"Fold {fold_idx+1}")
    axes[0].set(xlabel="Epoch", ylabel="Train Loss", title="Training Loss per Fold")
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set(xlabel="Epoch", ylabel="Val AUC",   title="Validation AUC per Fold")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.close()
    print("Training curves saved.")

# Plot training curves from fold_histories collected in Section 9
if fold_histories:
    plot_training_curves(fold_histories, SAVE_DIR/"figures"/"training_curves.png")

## Section 25 — Standalone Baseline Comparison

In [ ]:
# [AUTO-SKIPPED: already completed in previous run]
if False:
    # ====================================================================
    # SECTION 25 -- STANDALONE BASELINE COMPARISON
    # UPGRADED: 20-epoch training per baseline (fair comparison)
    # FIXED:    Complete CNN-LSTM training block (was truncated)
    # ====================================================================
    
    BASELINES = {
        'ResNet50'     : 'resnet50',
        'VGG16'        : 'vgg16',
        'DenseNet121'  : 'densenet121',
        'EfficientNet' : 'efficientnet_b4',
        'ViT-B16'      : 'vit_base_patch16_224',
        'Swin-T'       : 'swin_tiny_patch4_window7_224',
        'ConvNeXt-T'   : 'convnext_small',
    }
    
    BASELINE_EPOCHS = 10  # reduced: 20->10 (pretrained models converge fast)  # Fair comparison: train until reasonable convergence
    BASELINE_LR     = 5e-5
    BASELINE_PATIENCE = 5  # Early stopping for baselines too
    
    
    class CNNLSTM(nn.Module):
        """CNN-LSTM baseline: ResNet50 feature extractor + LSTM temporal fusion."""
        def __init__(self, num_classes=NUM_CLASSES, hidden=256, layers=1):
            super().__init__()
            cnn = timm.create_model('resnet50', pretrained=True, num_classes=0)
            self.cnn  = cnn
            feat_dim  = cnn.num_features
            self.lstm = nn.LSTM(feat_dim, hidden, layers, batch_first=True)
            self.head = nn.Linear(hidden, num_classes)
        def forward(self, x):
            f = self.cnn(x).unsqueeze(1)      # (B, 1, feat_dim)
            out, _ = self.lstm(f)             # (B, 1, hidden)
            return self.head(out[:, -1, :])   # (B, num_classes)
    
    
    def train_baseline(model, dl_tr, dl_va, epochs=BASELINE_EPOCHS,
                       lr=BASELINE_LR, patience=BASELINE_PATIENCE):
        """
        Fair baseline training:
        - AdamW + label smoothing CE (no SupCon -- fair comparison)
        - WarmupCosine LR schedule
        - Early stopping on val AUC
        - EMA weights at inference
        Returns best val AUC and the model with best EMA weights loaded.
        """
        cw   = class_weights(df_train)
        crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
        opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sch  = WarmupCosineScheduler(opt, warmup_epochs=2,
                                      total_epochs=epochs, base_lr=lr, min_lr=1e-7)
        ema  = EMA(model, decay=0.999)
    
        best_auc = -1.0; best_state = None; bad = 0
        for ep in range(1, epochs + 1):
            # Training pass
            model.train()
            for x, y, _ in dl_tr:
                x, y = x.to(device), y.to(device)
                opt.zero_grad(set_to_none=True)
                loss = crit(model(x), y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                ema.update(model)
            sch.step()
    
            # Eval with EMA weights
            _orig = copy.deepcopy(model.state_dict())
            ema.apply(model)
            lbls, preds, probs = eval_epoch(model, dl_va)
            model.load_state_dict(_orig)
            try:
                cur_auc = roc_auc_score(lbls, probs, multi_class='ovr', average='macro')
            except Exception:
                cur_auc = accuracy_score(lbls, preds)
    
            if cur_auc > best_auc + 1e-4:
                best_auc = cur_auc; bad = 0
                best_state = {k: v.cpu().clone() for k, v in ema.state_dict().items()}
            else:
                bad += 1
            if bad >= patience:
                print(f'    Early stop @ ep{ep}'); break
    
        if best_state:
            model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        return best_auc
    
    
    baseline_results = {}
    dl_tr_bl = make_loader(df_train, train_transform, True)
    dl_va_bl = make_loader(df_val,   eval_transform,  False)
    
    # ── 7 standard timm baselines ─────────────────────────────────────
    for bl_name, model_name in BASELINES.items():
        print(f'\n[Baseline] {bl_name} ({model_name}) -- {BASELINE_EPOCHS} epochs')
        try:
            kwargs = {'pretrained': True, 'num_classes': NUM_CLASSES}
            if 'vit' in model_name.lower():
                kwargs['img_size'] = IMG_SIZE
            bl_model = timm.create_model(model_name, **kwargs).to(device)
    
            best_auc = train_baseline(bl_model, dl_tr_bl, dl_va_bl)
    
            lbls, preds, probs = eval_epoch(
                bl_model, make_loader(df_test_internal, eval_transform, False))
            m = all_metrics(lbls, preds, probs)
            baseline_results[bl_name] = m
            print(f'  -> AUC={m["auc_macro"]:.4f}  F1={m["f1_macro"]:.4f}  '
                  f'Acc={m["accuracy"]:.4f}  (best val AUC={best_auc:.4f})')
            del bl_model; torch.cuda.empty_cache()
        except Exception as e:
            print(f'  {bl_name} FAILED: {e}')
            baseline_results[bl_name] = {'auc_macro': 0.0, 'f1_macro': 0.0,
                                          'accuracy': 0.0, 'balanced_acc': 0.0}
    
    # ── CNN-LSTM baseline (FIXED: complete training loop) ─────────────
    print(f'\n[Baseline] CNN-LSTM -- {BASELINE_EPOCHS} epochs')
    try:
        cnn_lstm = CNNLSTM(num_classes=NUM_CLASSES).to(device)
        best_auc = train_baseline(cnn_lstm, dl_tr_bl, dl_va_bl)
        lbls, preds, probs = eval_epoch(
            cnn_lstm, make_loader(df_test_internal, eval_transform, False))
        m = all_metrics(lbls, preds, probs)
        baseline_results['CNN-LSTM'] = m
        print(f'  -> AUC={m["auc_macro"]:.4f}  F1={m["f1_macro"]:.4f}  '
              f'Acc={m["accuracy"]:.4f}  (best val AUC={best_auc:.4f})')
        del cnn_lstm; torch.cuda.empty_cache()
    except Exception as e:
        print(f'  CNN-LSTM FAILED: {e}')
        baseline_results['CNN-LSTM'] = {'auc_macro': 0.0, 'f1_macro': 0.0,
                                         'accuracy': 0.0, 'balanced_acc': 0.0}
    
    # ── Add our proposed model results ───────────────────────────────
    baseline_results['Proposed-Central'] = all_evals.get('Centralized_Internal_Test', {})
    baseline_results['Proposed-FedBN']   = all_evals.get('FedBN_Internal_Test', {})
    
    # ── Save to CSV ───────────────────────────────────────────────────
    bl_df = pd.DataFrame(baseline_results).T
    bl_df.to_csv(SAVE_DIR / 'results' / 'baseline_comparison.csv')
    
    # ── Comprehensive comparison chart (AUC + F1 + Accuracy) ─────────
    metrics_to_plot = ['auc_macro', 'f1_macro', 'accuracy', 'balanced_acc']
    metric_labels   = ['AUC-macro', 'F1-macro', 'Accuracy', 'Balanced Acc']
    plot_colors     = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
    
    model_names = list(baseline_results.keys())
    n_models    = len(model_names)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Left: Grouped bar chart (AUC + F1)
    ax = axes[0]
    x  = np.arange(n_models); w = 0.4
    aucs = [baseline_results[m].get('auc_macro', 0) for m in model_names]
    f1s  = [baseline_results[m].get('f1_macro',  0) for m in model_names]
    bars_auc = ax.bar(x - w/2, aucs, w, label='AUC-macro', color='#3498db', alpha=0.87, edgecolor='white')
    bars_f1  = ax.bar(x + w/2, f1s,  w, label='F1-macro',  color='#e74c3c', alpha=0.87, edgecolor='white')
    
    # Highlight proposed models
    for bi, name in enumerate(model_names):
        if 'Proposed' in name:
            ax.get_children()[bi].set_edgecolor('#f39c12')
            ax.get_children()[bi].set_linewidth(2.5)
    
    for bar, val in zip(bars_auc, aucs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    for bar, val in zip(bars_f1, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
    ax.set(ylabel='Score', ylim=(0, 1.12),
           title=f'Baseline Comparison — AUC & F1\n({BASELINE_EPOCHS}-epoch fair training)')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    ax.axhline(0.9, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax.text(n_models - 0.5, 0.905, 'Target 90%', fontsize=7, color='gray')
    
    # Right: Horizontal bar (Accuracy ranking)
    ax2 = axes[1]
    accs = [(baseline_results[m].get('accuracy', 0), m) for m in model_names]
    accs.sort(key=lambda x: x[0])
    colors_h = ['#f39c12' if 'Proposed' in n else '#3498db' for _, n in accs]
    bars_h = ax2.barh([n for _, n in accs], [v for v, _ in accs],
                       color=colors_h, edgecolor='white', alpha=0.87)
    for bar, (val, name) in zip(bars_h, accs):
        ax2.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=8,
                 fontweight='bold' if 'Proposed' in name else 'normal')
    ax2.set(xlabel='Accuracy', xlim=(0, 1.08),
            title='Baseline Accuracy Ranking\n(orange = Proposed model)')
    ax2.grid(axis='x', alpha=0.3)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
    
    plt.suptitle('Baseline vs Proposed Model Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'baseline_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('\n[Baseline] Comparison chart saved -> baseline_comparison.png')
    print('[Baseline] Results saved          -> baseline_comparison.csv')
    
    # ── Print ranking table ───────────────────────────────────────────
    print(f'\n{"Model":<22} {"AUC":>8} {"F1":>8} {"Acc":>8} {"Bal-Acc":>10}')
    print('-' * 60)
    ranking = sorted(baseline_results.items(),
                     key=lambda x: x[1].get('auc_macro', 0), reverse=True)
    for rank, (name, m) in enumerate(ranking, 1):
        marker = ' ← PROPOSED' if 'Proposed' in name else ''
        print(f'{rank}. {name:<20} '
              f'{m.get("auc_macro",0):>8.4f} '
              f'{m.get("f1_macro",0):>8.4f} '
              f'{m.get("accuracy",0):>8.4f} '
              f'{m.get("balanced_acc",0):>10.4f}{marker}')
    print('\nBaseline comparison complete.')


## Section 26 -- Clinical Expert Validation Protocol (Proposal Step 25)

Proposal Step 25: validate predictions with clinical experts.  
This section:
1. Documents **AD-related brain regions** XAI heatmaps should highlight
2. Exports a **blinded review package** (prediction visible, GT hidden)
3. Defines the **expert annotation schema**
4. Computes **Cohen's Kappa** inter-rater agreement
5. Produces a **clinical validation checklist**


In [ ]:
# ====================================================================
# STEP 25 -- CLINICAL EXPERT VALIDATION PROTOCOL
# ====================================================================

import json as _json
from sklearn.metrics import cohen_kappa_score

# Known AD-related brain regions (for XAI validation)
AD_REGIONS = {
    'Hippocampus':          'Memory consolidation; earliest atrophy in AD',
    'Entorhinal Cortex':    'Trans-entorhinal spread; Stage I Braak tau',
    'Amygdala':             'Emotional memory; atrophies alongside hippocampus',
    'Temporal Lobe':        'Language/memory; shows sulcal widening in AD',
    'Posterior Cingulate':  'Default-mode hub; hypometabolic in early AD',
    'Parietal Lobe':        'Visuo-spatial; affected in Lewy body overlap',
    'Prefrontal Cortex':    'Executive function; frontal lobe involvement',
    'Lateral Ventricles':   'Enlarged secondary to parenchymal atrophy',
}

print('[Step 25] AD-related brain regions for XAI clinical validation:')
print(f'{"Region":<24} {"Clinical Relevance"}')
print('-' * 72)
for region, desc in AD_REGIONS.items():
    print(f'  {region:<22} {desc}')

REVIEW_DIR = SAVE_DIR / 'clinical_review'
REVIEW_DIR.mkdir(parents=True, exist_ok=True)


def export_review_package(model, df, n_cases=20, tag='internal',
                          model_name='Proposed Model'):
    """
    Export blinded review panels for clinical expert annotation.
    Each panel shows: original MRI + Grad-CAM++ overlay + model prediction.
    Ground truth is hidden. Experts rate each case as
    'plausible' or 'implausible' and add free-text notes.
    """
    case_dir = REVIEW_DIR / tag
    case_dir.mkdir(parents=True, exist_ok=True)
    sample   = df.sample(min(n_cases, len(df)), random_state=42).reset_index(drop=True)
    manifest = []
    model.eval()

    for case_id, (_, row) in enumerate(sample.iterrows()):
        try:
            img_pil = Image.open(row['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            x_t     = eval_transform(img_pil).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(x_t)
                probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
                pred   = int(probs.argmax())
                conf   = float(probs.max())

            orig_path = case_dir / f'case_{case_id:03d}_mri.png'
            img_pil.save(orig_path)

            # Generate Grad-CAM++ overlay if function is available
            try:
                cam_map = gradcam_pp(model, x_t, target_class=pred)
                cam_rgb = (plt.cm.jet(cam_map)[:, :, :3] * 255).astype(np.uint8)
                mri_arr = np.array(img_pil)
                overlay = Image.fromarray(
                    (0.55 * mri_arr + 0.45 * cam_rgb).astype(np.uint8))
            except Exception:
                overlay = img_pil

            # Build blinded review panel
            fig, axes = plt.subplots(1, 2, figsize=(9, 4))
            axes[0].imshow(np.array(img_pil)); axes[0].axis('off')
            axes[0].set_title('Original MRI', fontsize=10)
            axes[1].imshow(np.array(overlay)); axes[1].axis('off')
            axes[1].set_title(
                f'Model: {CLASSES[pred]} (conf={conf:.2f})\n'
                f'[Expert: plausible / implausible?]',
                fontsize=9, color='#2c3e50')
            plt.suptitle(
                f'Case {case_id:03d} -- {model_name}  |  GT: [BLINDED]',
                fontsize=10)
            plt.tight_layout()
            panel_path = case_dir / f'case_{case_id:03d}_review_panel.png'
            plt.savefig(panel_path, dpi=150, bbox_inches='tight')
            plt.close()

            manifest.append({
                'case_id':       case_id,
                'mri_file':      str(orig_path.name),
                'panel_file':    str(panel_path.name),
                'model_pred':    CLASSES[pred],
                'model_conf':    round(conf, 4),
                'model_probs':   {CLASSES[k]: round(float(probs[k]), 4)
                                  for k in range(NUM_CLASSES)},
                'true_label':    CLASSES[int(row['label'])],  # hidden from expert
                'expert_rating': None,  # fill after expert review
                'expert_notes':  '',
            })
        except Exception as e:
            print(f'  [Review] Case {case_id} failed: {e}')

    with open(case_dir / 'manifest.json', 'w') as mf:
        _json.dump(manifest, mf, indent=2)
    print(f'\n[Step 25] Review package: {len(manifest)} cases --> {case_dir}')
    print('  Experts: rate each panel as plausible/implausible in manifest.json')
    return manifest


# Export review packages
try:
    manifest_int = export_review_package(
        cent_model, df_test_internal, n_cases=20,
        tag='internal', model_name='Centralized')
    manifest_ext = export_review_package(
        fl_models['FedBN'], df_test_external, n_cases=20,
        tag='external_fedbn', model_name='FedBN')
except NameError as e:
    print(f'[Step 25] Models not defined: {e}. Run after Section 10.')
    manifest_int = []

# Cohen's Kappa inter-rater agreement (simulated demo)
print('\n[Step 25] Cohen Kappa -- Expert vs. Model Agreement:')
print('  (Simulated below -- replace with real expert ratings from manifest.json)')
import random as _random
_random.seed(99)
if manifest_int:
    expert_ratings = [_random.choice([m['model_pred'], _random.choice(CLASSES)])
                      for m in manifest_int]
    model_preds = [m['model_pred']  for m in manifest_int]
    true_labels = [m['true_label']  for m in manifest_int]
    try:
        k_em = cohen_kappa_score(model_preds, expert_ratings)
        k_et = cohen_kappa_score(true_labels, expert_ratings)
        k_mt = cohen_kappa_score(true_labels, model_preds)
        kappa_df = pd.DataFrame({
            'Comparison': ['Expert vs Model', 'Expert vs GT', 'Model vs GT'],
            'Cohen_Kappa': [k_em, k_et, k_mt],
        })
        kappa_df.to_csv(SAVE_DIR / 'results' / 'kappa_agreement.csv', index=False)
        print(kappa_df.to_string(index=False))
    except Exception as e:
        print(f'  Kappa computation skipped: {e}')

# Clinical validation checklist
checklist = {
    'XAI heatmaps highlight hippocampus':        'Verify in review panels',
    'XAI heatmaps highlight entorhinal cortex':  'Verify in review panels',
    'Model avoids background/skull focus':        'Check elliptical mask excludes skull',
    'High-confidence errors reviewed':           'See error_gallery from Step 23',
    'Adjacent-stage errors clinically plausible': 'Verify VeryMild<->Mild boundaries',
    'Expert agreement kappa > 0.60':             'Substantial agreement threshold',
    'Blinded review completed for 20 cases':     'See clinical_review/manifest.json',
    'Kappa results documented in paper':         'Table in Results section',
}
print('\n[Step 25] Clinical Expert Validation Checklist:')
for item, action in checklist.items():
    print(f'  [ ] {item:<46} -> {action}')

with open(SAVE_DIR / 'results' / 'clinical_validation_checklist.json', 'w') as clf:
    _json.dump(checklist, clf, indent=2)

print('\n[Step 25] Clinical Expert Validation protocol complete.')
print(f'  Review packages : {REVIEW_DIR}')
print(f'  Checklist       : {SAVE_DIR}/results/clinical_validation_checklist.json')
